# F2 - Preparação das justificativas R1, R2 e R3

Este notebook prepara, valida e consolida as justificativas explícitas utilizadas nas estratégias experimentais da F3. A F2 consome os artefatos congelados da F0 e da F1 e adiciona somente o componente textual produzido pelo modelo professor.

As justificativas possuem funções distintas:

- **R1** explica a passagem da intenção em linguagem natural para a IR;
- **R2** explica como a estrutura da IR é composta para formar Nile;
- **R3** explica diretamente a passagem da intenção em linguagem natural para Nile.

A entrada do CAMPI, a IR e a Nile permanecem em inglês ou em sua notação formal original. As instruções operacionais dirigidas ao modelo professor são documentadas em português, mas o conteúdo dos campos `r1`, `r2` e `r3` deve ser produzido integralmente em inglês. Essa regra evita introduzir troca de idioma nas demonstrações fornecidas posteriormente ao modelo aluno.

A geração textual é externa ao notebook. A F2 não simula o modelo professor nem substitui uma resposta ausente. Em vez disso, utiliza pontos de controle:

```text
preparação dos prompts
        ↓
arquivos do professor disponíveis? S/N
        ↓
carga e validação
        ↓
há reprovações?
        ↓
revisão seletiva pelo professor
        ↓
revalidação
        ↓
auditoria automática
        ↓
f2_operacional.zip
```

Quando a resposta ao ponto de controle for `N`, a etapa posterior é ignorada sem erro. Depois de adicionar os arquivos ao dataset do Kaggle, a execução pode ser retomada a partir do próprio ponto de controle.

A F2 preserva:

- os prompts enviados;
- as respostas brutas iniciais;
- os casos revisados;
- o feedback de validação;
- a versão consolidada de cada justificativa.

Não existe edição manual silenciosa. Justificativas reprovadas são encaminhadas individualmente ao mesmo modelo professor.

| Bloco | Etapa | Função metodológica | Resultado principal |
|---:|---|---|---|
| 1 | Configuração | Preparar ambiente, diretórios e parâmetros | Base da F2 |
| 2 | Auditoria | Confirmar F0 e F1 | Entradas validadas |
| 3 | Solicitações | Construir seis lotes externos | `f2_solicitacoes_professor.zip` |
| 4 | Ponto de controle | Carregar respostas quando disponíveis | Seis JSONs iniciais |
| 5 | Extração | Consolidar textos e preparar validação | `rationale_core.py` |
| 6 | R1 | Validar NL → IR | 50 resultados |
| 7 | R2 | Validar IR → Nile | 50 resultados |
| 8 | R3 | Validar NL → Nile | 50 resultados |
| 9 | Revisão | Corrigir somente casos reprovados | Revisões rastreáveis |
| 10 | Auditoria final | Conferir 50 exemplos e 150 justificativas | `f2_auditoria_final.csv` |
| 11 | Empacotamento | Congelar justificativas e histórico | `f2_operacional.zip` |

## Bloco 1 - Configuração inicial

### Objetivo

Este bloco prepara o ambiente da F2, define as quantidades esperadas e organiza os diretórios usados antes e depois da disponibilização das respostas do modelo professor.

### Procedimentos executados

O bloco:

- importa bibliotecas de sistema, serialização, compressão, hashing e manipulação tabular;
- configura a exibição e impede a inclusão de bytecodes;
- identifica o ambiente de execução;
- define os diretórios de entrada, trabalho, solicitações, revisões, saída operacional e pacote final;
- registra os caminhos dos artefatos da F2;
- fixa os 50 exemplos, três tipos de justificativa e total de 150 textos;
- define lotes de 25 exemplos, resultando em seis solicitações iniciais;
- registra os nomes exatos esperados para os arquivos de resposta;
- configura o modelo professor e a versão das justificativas;
- cria funções auxiliares de JSON, JSONL, GZIP, CSV, hashing, ZIP e exibição de tabelas;
- implementa a função de confirmação `S/N`.

### Separação das etapas

O diretório operacional final não é considerado concluído enquanto existirem respostas ou revisões pendentes. Antes disso, a F2 pode produzir somente:

- pacote de solicitações;
- pacote de revisões;
- arquivos temporários de auditoria.

### Padrão visual e terminológico

As tabelas seguem o mesmo padrão das fases anteriores. Rótulos administrativos são exibidos em português. Campos formais, operadores, unidades e valores da IR e da Nile permanecem em inglês.

### Resultado esperado

O ambiente estará preparado para localizar F0 e F1 e para construir as solicitações externas. Nenhum texto de justificativa é gerado neste bloco.

In [1]:
# ----------------------------------------------------------
# 1.1 Importação das bibliotecas principais
# ----------------------------------------------------------

import copy
import gzip
import hashlib
import importlib
import importlib.metadata as importlib_metadata
import importlib.util
import json
import os
import platform
import py_compile
import re
import shutil
import sys
import textwrap
import zipfile

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display


# ----------------------------------------------------------
# 1.2 Configuração de exibição e escrita de bytecode
# ----------------------------------------------------------

pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 220)
pd.set_option("display.width", 0)

sys.dont_write_bytecode = True


# ----------------------------------------------------------
# 1.3 Definição do ambiente e dos diretórios principais
# ----------------------------------------------------------

KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORKING_DIR = Path("/kaggle/working")

BASE_DIR = Path(
    os.environ.get(
        "F2_OUTPUT_ROOT",
        str(KAGGLE_WORKING_DIR if KAGGLE_WORKING_DIR.exists() else Path.cwd()),
    )
).resolve()

F2_WORK_DIR = BASE_DIR / "_f2_trabalho"
F2_FINAL_DIR = BASE_DIR / "f2_operacional"
REQUESTS_DIR = F2_WORK_DIR / "solicitacoes_professor"
REVISION_REQUESTS_DIR = F2_WORK_DIR / "revisoes_professor"
F0_EXTRACT_DIR = F2_WORK_DIR / "_f0_extraido"
F1_EXTRACT_DIR = F2_WORK_DIR / "_f1_extraido"
RESPONSES_EXTRACT_DIR = F2_WORK_DIR / "_respostas_extraidas"

F2_REQUESTS_ZIP_PATH = BASE_DIR / "f2_solicitacoes_professor.zip"
F2_REVISIONS_ZIP_PATH = BASE_DIR / "f2_revisoes_pendentes.zip"
F2_ZIP_PATH = BASE_DIR / "f2_operacional.zip"

for caminho in [F2_WORK_DIR, F2_FINAL_DIR]:
    if caminho.exists():
        shutil.rmtree(caminho)

for caminho in [F2_REQUESTS_ZIP_PATH, F2_REVISIONS_ZIP_PATH, F2_ZIP_PATH]:
    if caminho.exists():
        caminho.unlink()

REQUESTS_DIR.mkdir(parents=True, exist_ok=True)
REVISION_REQUESTS_DIR.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------
# 1.4 Definição dos artefatos finais da F2
# ----------------------------------------------------------

RATIONALE_CORE_PATH = F2_WORK_DIR / "rationale_core.py"
RATIONALES_REFERENCES_PATH = F2_FINAL_DIR / "rationales_references.jsonl"
PROFESSOR_GENERATIONS_PATH = F2_FINAL_DIR / "professor_generations.jsonl.gz"
RATIONALE_VALIDATION_PATH = F2_FINAL_DIR / "rationale_validation.csv"
FINAL_RATIONALE_CORE_PATH = F2_FINAL_DIR / "rationale_core.py"
MANIFEST_PATH = F2_FINAL_DIR / "manifest.json"

REQUESTS_INDEX_PATH = REQUESTS_DIR / "indice_solicitacoes.csv"
REQUESTS_MANIFEST_PATH = REQUESTS_DIR / "manifest_solicitacoes.json"
REVISION_INDEX_PATH = REVISION_REQUESTS_DIR / "indice_revisoes.csv"
REVISION_FEEDBACK_PATH = REVISION_REQUESTS_DIR / "feedback_revisoes.jsonl"


# ----------------------------------------------------------
# 1.5 Parâmetros fixos da fase
# ----------------------------------------------------------

FASE = "F2"
DATASET_ID = "CAMPI"
EXPECTED_EXAMPLES = 50
RATIONALE_TYPES = ("R1", "R2", "R3")
EXPECTED_RATIONALES = EXPECTED_EXAMPLES * len(RATIONALE_TYPES)
BATCH_SIZE = 25
EXPECTED_BATCHES_PER_TYPE = EXPECTED_EXAMPLES // BATCH_SIZE
EXPECTED_GENERATION_BATCHES = EXPECTED_BATCHES_PER_TYPE * len(RATIONALE_TYPES)
TEACHER_MODEL = "DeepSeek-V4"
RATIONALE_VERSION = "1.4.0"
EXPECTED_FINAL_FILES = 5

if EXPECTED_EXAMPLES % BATCH_SIZE != 0:
    raise ValueError("BATCH_SIZE deve dividir EXPECTED_EXAMPLES exatamente.")


# ----------------------------------------------------------
# 1.6 Funções auxiliares de leitura, escrita e hash
# ----------------------------------------------------------

def salvar_json(objeto, caminho: Path, indent: int = 2) -> None:
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    with caminho.open("w", encoding="utf-8") as arquivo:
        json.dump(objeto, arquivo, ensure_ascii=False, indent=indent)


def ler_json(caminho: Path):
    with Path(caminho).open("r", encoding="utf-8") as arquivo:
        return json.load(arquivo)


def salvar_jsonl(registros, caminho: Path) -> None:
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    with caminho.open("w", encoding="utf-8") as arquivo:
        for registro in registros:
            arquivo.write(json.dumps(registro, ensure_ascii=False) + "\n")


def ler_jsonl(caminho: Path):
    registros = []
    with Path(caminho).open("r", encoding="utf-8") as arquivo:
        for numero_linha, linha in enumerate(arquivo, start=1):
            linha = linha.strip()
            if not linha:
                continue
            try:
                registros.append(json.loads(linha))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"JSON inválido em {caminho}, linha {numero_linha}: {exc}"
                ) from exc
    return registros


def salvar_jsonl_gzip(registros, caminho: Path) -> None:
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(caminho, "wt", encoding="utf-8") as arquivo:
        for registro in registros:
            arquivo.write(json.dumps(registro, ensure_ascii=False) + "\n")


def salvar_texto(texto: str, caminho: Path) -> None:
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(str(texto), encoding="utf-8")


def ler_texto(caminho: Path) -> str:
    caminho = Path(caminho)
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo de texto não encontrado: {caminho}")
    return caminho.read_text(encoding="utf-8")


def calcular_sha256(caminho: Path) -> str:
    digest = hashlib.sha256()
    with Path(caminho).open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
            digest.update(bloco)
    return digest.hexdigest()


def calcular_sha256_texto(texto: str) -> str:
    return hashlib.sha256(str(texto).encode("utf-8")).hexdigest()


def versao_pacote(nome: str) -> str:
    try:
        return importlib_metadata.version(nome)
    except importlib_metadata.PackageNotFoundError:
        return "nao_instalado"


def carregar_modulo_python(caminho: Path, nome_modulo: str):
    caminho = Path(caminho)
    if not caminho.is_file():
        raise FileNotFoundError(f"Módulo Python não encontrado: {caminho}")

    especificacao = importlib.util.spec_from_file_location(nome_modulo, caminho)
    if especificacao is None or especificacao.loader is None:
        raise ImportError(f"Não foi possível carregar o módulo: {caminho}")

    modulo = importlib.util.module_from_spec(especificacao)
    sys.modules[nome_modulo] = modulo
    especificacao.loader.exec_module(modulo)
    return modulo


# ----------------------------------------------------------
# 1.7 Funções auxiliares de JSON e templates
# ----------------------------------------------------------

def json_legivel(objeto) -> str:
    return json.dumps(objeto, ensure_ascii=False, indent=2)


def json_compacto(objeto) -> str:
    return json.dumps(
        objeto,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )


def identificar_placeholders(template: str):
    return sorted(set(re.findall(r"\{\{\s*([A-Za-z0-9_]+)\s*\}\}", str(template))))


def preencher_template(template: str, substituicoes: dict) -> str:
    texto = str(template)
    for chave, valor in substituicoes.items():
        padrao = r"\{\{\s*" + re.escape(str(chave)) + r"\s*\}\}"
        texto = re.sub(padrao, str(valor), texto)

    restantes = identificar_placeholders(texto)
    if restantes:
        raise ValueError(
            "O template ainda contém placeholders não preenchidos: "
            + ", ".join(restantes)
        )
    return texto


def extrair_json_de_resposta(conteudo: str):
    texto = str(conteudo).strip()
    if not texto:
        raise ValueError("A resposta do modelo professor está vazia.")

    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        pass

    blocos = re.findall(
        r"```(?:json)?\s*(.*?)```",
        texto,
        flags=re.DOTALL | re.IGNORECASE,
    )
    for bloco in blocos:
        try:
            return json.loads(bloco.strip())
        except json.JSONDecodeError:
            continue

    inicio = texto.find("{")
    fim = texto.rfind("}")
    if inicio >= 0 and fim > inicio:
        try:
            return json.loads(texto[inicio: fim + 1])
        except json.JSONDecodeError:
            pass

    raise ValueError("Não foi possível extrair um objeto JSON válido da resposta.")


# ----------------------------------------------------------
# 1.8 Funções auxiliares de arquivos e ZIP
# ----------------------------------------------------------

def caminho_esta_dentro(pai: Path, filho: Path) -> bool:
    try:
        Path(filho).resolve().relative_to(Path(pai).resolve())
        return True
    except ValueError:
        return False


def criar_zip(arquivo_zip: Path, itens) -> Path:
    arquivo_zip = Path(arquivo_zip)
    arquivo_zip.parent.mkdir(parents=True, exist_ok=True)
    if arquivo_zip.exists():
        arquivo_zip.unlink()

    with zipfile.ZipFile(arquivo_zip, "w", compression=zipfile.ZIP_DEFLATED) as pacote:
        for origem, nome_interno in itens:
            origem = Path(origem)
            if not origem.is_file():
                raise FileNotFoundError(f"Arquivo ausente durante o empacotamento: {origem}")
            pacote.write(origem, arcname=str(nome_interno))

    return arquivo_zip


def extrair_zip(caminho_zip: Path, destino: Path) -> Path:
    caminho_zip = Path(caminho_zip)
    destino = Path(destino)
    if destino.exists():
        shutil.rmtree(destino)
    destino.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(caminho_zip, "r") as pacote:
        pacote.extractall(destino)
    return destino


# ----------------------------------------------------------
# 1.9 Exibição padronizada e numeração das tabelas
# ----------------------------------------------------------

CONTADOR_TABELAS = 0

ROTULOS_COLUNAS = {
    "fase": "fase",
    "dataset": "conjunto de dados",
    "origem": "origem",
    "tipo_artefato": "tipo de artefato",
    "arquivo": "arquivo",
    "existe": "existe",
    "hash_registrado": "hash registrado",
    "hash_observado": "hash observado",
    "hash_confere": "hash confere",
    "status": "status",
    "registros": "registros",
    "ids_unicos": "IDs únicos",
    "referencias_validas": "referências válidas",
    "irs_validas": "IRs válidas",
    "id": "ID",
    "university": "universidade",
    "nl": "entrada em linguagem natural",
    "nile": "Nile",
    "scope_type": "scope type",
    "operators": "operators",
    "has_temporal": "possui temporal constraint",
    "tipo_justificativa": "tipo de justificativa",
    "idioma_prompt": "idioma do prompt",
    "idioma_justificativa": "idioma da justificativa",
    "lote": "lote",
    "request_id": "ID da solicitação",
    "total_exemplos": "exemplos",
    "primeiro_id": "primeiro ID",
    "ultimo_id": "último ID",
    "arquivo_prompt": "arquivo do prompt",
    "arquivo_resposta": "arquivo esperado da resposta",
    "caracteres_prompt": "caracteres do prompt",
    "sha256_prompt": "SHA-256 do prompt",
    "resposta_encontrada": "resposta encontrada",
    "caminho_resposta": "caminho da resposta",
    "itens_extraidos": "itens extraídos",
    "r1_caracteres": "caracteres de R1",
    "r2_caracteres": "caracteres de R2",
    "r3_caracteres": "caracteres de R3",
    "aprovada": "aprovada",
    "aprovadas": "aprovadas",
    "reprovadas": "reprovadas",
    "erros": "erros",
    "quantidade_erros": "quantidade de erros",
    "quantidade_avisos": "quantidade de avisos",
    "ancoras_encontradas": "âncoras encontradas",
    "ancoras_esperadas": "âncoras esperadas",
    "cobertura_ancoras": "cobertura de âncoras",
    "caracteres": "caracteres",
    "revisao_necessaria": "revisão necessária",
    "revisao_encontrada": "revisão encontrada",
    "revisao_aplicada": "revisão aplicada",
    "aprovada_inicialmente": "aprovada inicialmente",
    "aprovada_finalmente": "aprovada finalmente",
    "total_justificativas": "justificativas",
    "geracoes_professor": "gerações do professor",
    "revisoes_aplicadas": "revisões aplicadas",
    "arquivos_no_zip": "arquivos no ZIP",
    "zip": "ZIP",
}

ROTULOS_VALORES = {
    "initial_generation": "geração inicial",
    "revision": "revisão",
    "pending": "pendente",
    "ready": "pronto",
    "approved": "aprovado",
    "rejected": "reprovado",
    "R1": "R1",
    "R2": "R2",
    "R3": "R3",
}


def traduzir_valores_para_exibicao(df: pd.DataFrame) -> pd.DataFrame:
    df_exibicao = df.copy()

    def converter_valor(valor):
        if isinstance(valor, (bool, np.bool_)):
            return "sim" if valor else "não"
        if valor is None:
            return "-"
        try:
            if not isinstance(valor, str) and pd.isna(valor):
                return "-"
        except Exception:
            pass
        if isinstance(valor, str):
            valor_limpo = valor.strip()
            valor_lower = valor_limpo.lower()
            if valor_lower == "true":
                return "sim"
            if valor_lower == "false":
                return "não"
            if valor_lower in {
                "nan", "none", "null", "nat", "n/a",
                "não se aplica", "nao se aplica",
            }:
                return "-"
            return ROTULOS_VALORES.get(valor_limpo, valor)
        return valor

    for coluna in df_exibicao.columns:
        df_exibicao[coluna] = df_exibicao[coluna].apply(converter_valor)

    return df_exibicao.rename(
        columns={
            coluna: ROTULOS_COLUNAS.get(coluna, str(coluna).replace("_", " "))
            for coluna in df_exibicao.columns
        }
    )


def exibir_tabela(
    df: pd.DataFrame,
    titulo: str = None,
    altura_px: int = None,
    largura_px: int = None,
    mostrar_indice: bool = False,
) -> None:
    global CONTADOR_TABELAS

    if df is None:
        print("Tabela não disponível.")
        return
    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame(df)
    if df.empty:
        print("Tabela vazia.")
        return

    CONTADOR_TABELAS += 1
    tabela = traduzir_valores_para_exibicao(df)

    titulo_final = f"Tabela {CONTADOR_TABELAS}"
    if titulo:
        titulo_final += f". {titulo}"

    html_tabela = tabela.to_html(
        escape=True,
        index=mostrar_indice,
        border=0,
        justify="left",
        classes="tabela_saida",
    )

    largura_css = f"{largura_px}px" if largura_px is not None else "100%"
    altura_css = (
        "overflow-y: visible;"
        if altura_px is None
        else f"max-height: {altura_px}px; overflow-y: auto;"
    )

    display(HTML(f"""
    <div style="
        font-weight: 600;
        font-size: 15px;
        margin-top: 8px;
        margin-bottom: 6px;
        color: #f1f1f1;
    ">
        {titulo_final}
    </div>

    <div class="container_tabela_saida" style="
        display: inline-block;
        width: {largura_css};
        max-width: 100%;
        overflow-x: auto;
        {altura_css}
        box-sizing: border-box;
        padding: 0;
        margin-top: 8px;
        margin-bottom: 12px;
        border: none;
        border-radius: 0;
    ">
        <style>
            .container_tabela_saida {{ box-sizing: border-box !important; }}
            .container_tabela_saida table.tabela_saida {{
                border-collapse: collapse !important;
                table-layout: auto !important;
                width: auto !important;
                min-width: unset !important;
                max-width: none !important;
                font-family: Arial, sans-serif !important;
                font-size: 13px !important;
                background-color: #111 !important;
                color: #f1f1f1 !important;
                border: 1px solid #555 !important;
            }}
            .container_tabela_saida table.tabela_saida thead th {{
                position: sticky !important;
                top: 0 !important;
                z-index: 2 !important;
                background-color: #2b2b2b !important;
                color: #ffffff !important;
                font-weight: bold !important;
                padding: 7px !important;
                text-align: left !important;
                white-space: nowrap !important;
                border: 1px solid #777 !important;
                border-bottom: 2px solid #888 !important;
            }}
            .container_tabela_saida table.tabela_saida tbody td {{
                padding: 7px !important;
                vertical-align: top !important;
                text-align: left !important;
                white-space: nowrap !important;
                border: 1px solid #555 !important;
            }}
            .container_tabela_saida table.tabela_saida tbody tr:nth-child(even) td {{
                background-color: #1b1b1b !important;
            }}
            .container_tabela_saida table.tabela_saida tbody tr:nth-child(odd) td {{
                background-color: #111 !important;
            }}
        </style>
        {html_tabela}
    </div>
    """))




# ----------------------------------------------------------
# 1.10 Controle interativo da execução
# ----------------------------------------------------------

def solicitar_sim_nao(pergunta: str, variavel_ambiente: str | None = None) -> bool:
    """
    Solicita uma confirmação S/N.

    A variável de ambiente é opcional e existe apenas para execuções
    automatizadas ou testes. No uso normal do Kaggle, a resposta é
    informada diretamente no campo exibido pelo notebook.
    """
    valor_ambiente = None
    if variavel_ambiente:
        valor_ambiente = os.environ.get(variavel_ambiente)

    if valor_ambiente is not None:
        resposta = str(valor_ambiente).strip().lower()
        print(f"{pergunta} {valor_ambiente}")
    else:
        resposta = input(pergunta).strip().lower()

    while resposta not in {"s", "sim", "n", "nao", "não"}:
        print("Resposta inválida. Digite S para sim ou N para não.")
        resposta = input(pergunta).strip().lower()

    return resposta in {"s", "sim"}


F2_CONTINUAR_VALIDACAO = False
F2_CONTINUAR_FINALIZACAO = False
F2_AUDITORIA_APROVADA = False
F2_AUDITORIA_CONFIRMADA_EM_UTC = None


# ----------------------------------------------------------
# 1.11 Saída do bloco
# ----------------------------------------------------------

print("Bloco 1 concluído")
print(f"Fase: {FASE}")
print(f"Conjunto de dados: {DATASET_ID}")
print(f"Modelo professor definido: {TEACHER_MODEL}")
print(f"Justificativas esperadas: {EXPECTED_RATIONALES}")
print(f"Tamanho fixo dos lotes: {BATCH_SIZE}")
print(f"Diretório temporário: {F2_WORK_DIR}")
print("Status: OK")

Bloco 1 concluído
Fase: F2
Conjunto de dados: CAMPI
Modelo professor definido: DeepSeek-V4
Justificativas esperadas: 150
Tamanho fixo dos lotes: 25
Diretório temporário: /kaggle/working/_f2_trabalho
Status: OK


## Bloco 2 - Carga e auditoria dos artefatos da F0 e da F1

### Objetivo

Este bloco confirma que as justificativas serão produzidas a partir das versões corretas do CAMPI, da Nile e da IR.

### Localização das entradas

Os diretórios operacionais de F0 e F1 são procurados nos datasets montados pelo Kaggle e nos caminhos informados pelas variáveis de ambiente. A identificação depende dos manifestos e dos arquivos obrigatórios.

### Auditoria da F0

São verificados:

- manifesto da F0;
- `campi_canonical.csv`;
- gramática e núcleo Nile;
- validação das referências;
- templates necessários à F2;
- hashes registrados.

### Auditoria da F1

São verificados:

- manifesto da F1;
- `ir_schema.json`;
- `ir_core.py`;
- `ir_references.jsonl`;
- `ir_validation.csv`;
- vínculo da F1 com a mesma versão da F0.

### Conferências cruzadas

O bloco confirma:

- 50 IDs na F0;
- 50 IRs na F1;
- correspondência exata dos IDs;
- 50 referências Nile válidas;
- 50 IRs válidas pelo schema;
- equivalência registrada na F1;
- ausência de divergência entre hashes e manifestos.

### Importação controlada

Os módulos formais são carregados a partir dos artefatos congelados, sem alterar seus arquivos.

### Resultado esperado

As tabelas apresentam a origem e a integridade das entradas. Qualquer arquivo ausente, hash divergente ou incompatibilidade interrompe a preparação das solicitações.

In [2]:
# ----------------------------------------------------------
# 2.1 Localização de uma base de artefatos
# ----------------------------------------------------------

def localizar_base_artefatos(
    env_var: str,
    marcador: str,
    arquivos_obrigatorios,
    nomes_zip,
    destino_extracao: Path,
) -> Path:
    caminho_env = os.environ.get(env_var)
    if caminho_env:
        caminho = Path(caminho_env).expanduser().resolve()
        if caminho.is_file() and caminho.suffix.lower() == ".zip":
            extrair_zip(caminho, destino_extracao)
            caminho = destino_extracao
        if caminho.is_dir():
            candidatos = [caminho] + [p.parent for p in caminho.rglob(marcador)]
            for candidato in candidatos:
                if all((candidato / nome).is_file() for nome in arquivos_obrigatorios):
                    return candidato.resolve()
        raise FileNotFoundError(
            f"A variável {env_var} não aponta para uma base válida: {caminho}"
        )

    raizes = [KAGGLE_INPUT_DIR, Path.cwd()]
    candidatos = []

    for raiz in raizes:
        if not raiz.exists():
            continue
        for arquivo_marcador in raiz.rglob(marcador):
            if caminho_esta_dentro(F2_WORK_DIR, arquivo_marcador):
                continue
            base = arquivo_marcador.parent
            if all((base / nome).is_file() for nome in arquivos_obrigatorios):
                candidatos.append(base.resolve())

    candidatos = sorted(set(candidatos), key=lambda p: (len(str(p)), str(p)))
    if candidatos:
        return candidatos[0]

    for raiz in raizes:
        if not raiz.exists():
            continue
        for nome_zip in nomes_zip:
            for caminho_zip in raiz.rglob(nome_zip):
                if caminho_esta_dentro(F2_WORK_DIR, caminho_zip):
                    continue
                extrair_zip(caminho_zip, destino_extracao)
                candidatos_extraidos = [
                    p.parent
                    for p in destino_extracao.rglob(marcador)
                    if all((p.parent / nome).is_file() for nome in arquivos_obrigatorios)
                ]
                if candidatos_extraidos:
                    return sorted(
                        candidatos_extraidos,
                        key=lambda p: (len(str(p)), str(p)),
                    )[0].resolve()

    raise FileNotFoundError(
        f"Não foi possível localizar a base que contém: {', '.join(arquivos_obrigatorios)}"
    )


# ----------------------------------------------------------
# 2.2 Definição dos arquivos obrigatórios
# ----------------------------------------------------------

ARQUIVOS_F0_OBRIGATORIOS = [
    "manifest.json",
    "campi_canonical.csv",
    "validation_references.csv",
    "biblioteca_prompts.json",
    "prompts/f2_geracao_r1.txt",
    "prompts/f2_geracao_r2.txt",
    "prompts/f2_geracao_r3.txt",
    "prompts/f2_revisao_justificativa_reprovada.txt",
]

ARQUIVOS_F1_OBRIGATORIOS = [
    "manifest.json",
    "ir_references.jsonl",
    "ir_schema.json",
    "ir_core.py",
    "ir_validation.csv",
]

F0_DIR = localizar_base_artefatos(
    env_var="F0_ARTIFACTS_DIR",
    marcador="campi_canonical.csv",
    arquivos_obrigatorios=ARQUIVOS_F0_OBRIGATORIOS,
    nomes_zip=["f0_operacional.zip", "f0_operacional*.zip"],
    destino_extracao=F0_EXTRACT_DIR,
)

F1_DIR = localizar_base_artefatos(
    env_var="F1_ARTIFACTS_DIR",
    marcador="ir_references.jsonl",
    arquivos_obrigatorios=ARQUIVOS_F1_OBRIGATORIOS,
    nomes_zip=["f1_operacional.zip", "f1_operacional*.zip"],
    destino_extracao=F1_EXTRACT_DIR,
)


# ----------------------------------------------------------
# 2.3 Carga dos manifestos e validação de identidade
# ----------------------------------------------------------

manifesto_f0 = ler_json(F0_DIR / "manifest.json")
manifesto_f1 = ler_json(F1_DIR / "manifest.json")

if manifesto_f0.get("fase") != "F0":
    raise ValueError("O manifesto localizado para a F0 não identifica a fase F0.")
if manifesto_f1.get("fase") != "F1":
    raise ValueError("O manifesto localizado para a F1 não identifica a fase F1.")
if manifesto_f0.get("dataset", {}).get("id") != DATASET_ID:
    raise ValueError("O manifesto da F0 não identifica o conjunto de dados CAMPI.")
if manifesto_f1.get("ir", {}).get("total_examples") != EXPECTED_EXAMPLES:
    raise ValueError("O manifesto da F1 não registra 50 IRs.")


# ----------------------------------------------------------
# 2.4 Auditoria dos hashes registrados
# ----------------------------------------------------------

def auditar_manifesto(base: Path, manifesto: dict, fase: str, obrigatorios):
    registros_manifesto = {
        str(item["arquivo"]): item
        for item in manifesto.get("files", [])
    }

    linhas = []
    erros = []

    for nome_relativo in obrigatorios:
        caminho = base / nome_relativo
        existe = caminho.is_file()
        registro = registros_manifesto.get(nome_relativo)
        hash_registrado = None if registro is None else registro.get("sha256")
        hash_observado = calcular_sha256(caminho) if existe else None
        hash_confere = (
            existe
            and hash_registrado is not None
            and hash_observado == hash_registrado
        )

        linhas.append({
            "fase": fase,
            "arquivo": nome_relativo,
            "existe": existe,
            "hash_registrado": hash_registrado,
            "hash_observado": hash_observado,
            "hash_confere": hash_confere,
        })

        if not existe:
            erros.append(f"{fase}: arquivo ausente: {nome_relativo}")
        elif nome_relativo != "manifest.json" and registro is None:
            erros.append(f"{fase}: arquivo sem registro no manifesto: {nome_relativo}")
        elif nome_relativo != "manifest.json" and not hash_confere:
            erros.append(f"{fase}: hash divergente: {nome_relativo}")

    return linhas, erros


auditoria_f0, erros_f0 = auditar_manifesto(
    F0_DIR,
    manifesto_f0,
    "F0",
    ARQUIVOS_F0_OBRIGATORIOS,
)
auditoria_f1, erros_f1 = auditar_manifesto(
    F1_DIR,
    manifesto_f1,
    "F1",
    ARQUIVOS_F1_OBRIGATORIOS,
)

# O próprio manifesto não registra o seu hash dentro de si.
for linha in auditoria_f0 + auditoria_f1:
    if linha["arquivo"] == "manifest.json":
        linha["hash_confere"] = True

if erros_f0 or erros_f1:
    raise ValueError("Falhas na auditoria:\n- " + "\n- ".join(erros_f0 + erros_f1))


# ----------------------------------------------------------
# 2.5 Verificação do vínculo entre F0 e F1
# ----------------------------------------------------------

hash_campi_f0 = calcular_sha256(F0_DIR / "campi_canonical.csv")
hash_grammar_f0 = calcular_sha256(F0_DIR / "nile_subset.lark") if (F0_DIR / "nile_subset.lark").is_file() else None
hash_nile_core_f0 = calcular_sha256(F0_DIR / "nile_core.py") if (F0_DIR / "nile_core.py").is_file() else None

vinculo_f1 = manifesto_f1.get("input", {})

if vinculo_f1.get("campi_canonical_sha256") != hash_campi_f0:
    raise ValueError("A F1 não foi construída a partir do campi_canonical.csv conectado.")
if hash_grammar_f0 and vinculo_f1.get("grammar_sha256") != hash_grammar_f0:
    raise ValueError("A F1 não está vinculada à gramática conectada da F0.")
if hash_nile_core_f0 and vinculo_f1.get("nile_core_sha256") != hash_nile_core_f0:
    raise ValueError("A F1 não está vinculada ao nile_core.py conectado da F0.")


# ----------------------------------------------------------
# 2.6 Carga e integração dos 50 registros
# ----------------------------------------------------------

campi_df = pd.read_csv(F0_DIR / "campi_canonical.csv")
validacao_referencias_df = pd.read_csv(F0_DIR / "validation_references.csv")
ir_registros = ler_jsonl(F1_DIR / "ir_references.jsonl")
ir_schema = ler_json(F1_DIR / "ir_schema.json")
ir_core = carregar_modulo_python(F1_DIR / "ir_core.py", "f2_ir_core")

if len(campi_df) != EXPECTED_EXAMPLES:
    raise ValueError(f"O CAMPI canônico deveria conter {EXPECTED_EXAMPLES} registros.")
if campi_df["id"].nunique() != EXPECTED_EXAMPLES:
    raise ValueError("O CAMPI canônico contém IDs ausentes ou duplicados.")
if len(ir_registros) != EXPECTED_EXAMPLES:
    raise ValueError(f"A F1 deveria conter {EXPECTED_EXAMPLES} IRs.")

ir_por_id = {}
for registro in ir_registros:
    item_id = str(registro.get("id", "")).strip()
    if not item_id:
        raise ValueError("Foi encontrada uma IR sem ID.")
    if item_id in ir_por_id:
        raise ValueError(f"IR duplicada para o ID: {item_id}")
    resultado_ir = ir_core.validate_ir(registro.get("ir"), ir_schema)
    if not resultado_ir["schema_valid"]:
        raise ValueError(f"IR inválida para {item_id}: {resultado_ir['schema_errors']}")
    ir_por_id[item_id] = registro["ir"]

ids_campi = set(campi_df["id"].astype(str))
ids_ir = set(ir_por_id)
if ids_campi != ids_ir:
    raise ValueError("Os IDs do CAMPI e das IRs não coincidem integralmente.")

coluna_validade = "valid" if "valid" in validacao_referencias_df.columns else None
if coluna_validade and not validacao_referencias_df[coluna_validade].astype(bool).all():
    raise ValueError("Existem referências Nile não aprovadas na F0.")

registros_base = []
for linha in campi_df.sort_values("id").itertuples(index=False):
    registros_base.append({
        "id": str(linha.id),
        "university": str(linha.university),
        "nl": str(linha.nl),
        "nile": str(linha.nile_canonical),
        "ir": ir_por_id[str(linha.id)],
    })


# ----------------------------------------------------------
# 2.7 Exibição da auditoria e da base integrada
# ----------------------------------------------------------

resumo_bases_df = pd.DataFrame([
    {
        "fase": "F0",
        "origem": str(F0_DIR),
        "registros": len(campi_df),
        "ids_unicos": campi_df["id"].nunique(),
        "referencias_validas": int(len(validacao_referencias_df)),
        "status": "approved",
    },
    {
        "fase": "F1",
        "origem": str(F1_DIR),
        "registros": len(ir_registros),
        "ids_unicos": len(ir_por_id),
        "irs_validas": len(ir_por_id),
        "status": "approved",
    },
])

exibir_tabela(
    resumo_bases_df,
    titulo="Resumo das bases carregadas",
)

exibir_tabela(
    pd.DataFrame(auditoria_f0 + auditoria_f1),
    titulo="Auditoria dos arquivos obrigatórios",
    altura_px=500,
)

amostra_integrada_df = pd.DataFrame([
    {
        "id": item["id"],
        "university": item["university"],
        "nl": item["nl"],
        "nile": item["nile"],
        "scope_type": item["ir"]["scope"]["type"],
        "operators": " | ".join(op["operator"] for op in item["ir"]["operations"]),
        "has_temporal": item["ir"].get("temporal_constraint") is not None,
    }
    for item in registros_base
])

exibir_tabela(
    amostra_integrada_df,
    titulo="Registros integrados que serão enviados ao modelo professor",
    altura_px=520,
)

print("Bloco 2 concluído")
print(f"F0 localizada em: {F0_DIR}")
print(f"F1 localizada em: {F1_DIR}")
print(f"Registros integrados: {len(registros_base)}")
print("Status: OK")

fase,origem,registros,IDs únicos,referências válidas,status,IRs válidas
F0,/kaggle/input/datasets/thiagoarajoguedes/f0-operacional,50,50,50.0,aprovado,-
F1,/kaggle/input/datasets/thiagoarajoguedes/f1-operacional,50,50,-,aprovado,50.0


fase,arquivo,existe,hash registrado,hash observado,hash confere
F0,manifest.json,sim,-,53f33d7efd81d2f63243c467e6f67c151f52f4c330935cb13488ba05f8ba4f3d,sim
F0,campi_canonical.csv,sim,6cd2f7fa0fcd5a8b35529cb0a3249217d53d51a642d96b743231f711f1cb3b1d,6cd2f7fa0fcd5a8b35529cb0a3249217d53d51a642d96b743231f711f1cb3b1d,sim
F0,validation_references.csv,sim,184fd4e52c44b25f20c05d2a29ffbb1bef3e1fb1feca885c8283ec18c113d37e,184fd4e52c44b25f20c05d2a29ffbb1bef3e1fb1feca885c8283ec18c113d37e,sim
F0,biblioteca_prompts.json,sim,14276ebd3ff20aad7ca29e0f61a2a7c27b5a4eb374a17b528705cc79559b9a9d,14276ebd3ff20aad7ca29e0f61a2a7c27b5a4eb374a17b528705cc79559b9a9d,sim
F0,prompts/f2_geracao_r1.txt,sim,aa6502549e55be222eefbc996670be8e76761973a9868ee4429a9b6072266b35,aa6502549e55be222eefbc996670be8e76761973a9868ee4429a9b6072266b35,sim
F0,prompts/f2_geracao_r2.txt,sim,10c6d733f92c095d07679e0cca54ed969d6b2ece040b9223afcca19d02cdb696,10c6d733f92c095d07679e0cca54ed969d6b2ece040b9223afcca19d02cdb696,sim
F0,prompts/f2_geracao_r3.txt,sim,4d3edb143bf185a4eebfaa36b1754d4768150d01e05e6fb1c6bcdf6e58947f68,4d3edb143bf185a4eebfaa36b1754d4768150d01e05e6fb1c6bcdf6e58947f68,sim
F0,prompts/f2_revisao_justificativa_reprovada.txt,sim,42f797f664d7250d1393a5b09761d8c40b9b6584ad85d18a72c7aaf8cf1b8166,42f797f664d7250d1393a5b09761d8c40b9b6584ad85d18a72c7aaf8cf1b8166,sim
F1,manifest.json,sim,-,e15ad59c7161dce8bb93ebe7cd3cae38199b869ba724cb87aef2eddc4f043df9,sim
F1,ir_references.jsonl,sim,1572888aadd6cb05798c6ba3ea1c8a84c9c6110199275d77d45566a93330253c,1572888aadd6cb05798c6ba3ea1c8a84c9c6110199275d77d45566a93330253c,sim


ID,universidade,entrada em linguagem natural,Nile,scope type,operators,possui temporal constraint
campi_001,University of Illinois - Urbana Champaign,"If a student is in obvious violations of copyright law by using a room's wired connection (ie ResNet) to distribute copyrighted materials, the room's connection will be disabled, and the issue could be sent to Housing Student Judicial Affairs",define intent uniIntent: for group('students') add middlebox('copyright monitoring'),for,add,não
campi_002,University of Illinois - Urbana Champaign,"Currently, the University of Illinois does not have any rate limits",define intent uniIntent: for endpoint('university') unset bandwidth(),for,unset,não
campi_003,University of Illinois - Urbana Champaign,University Housing monitors only the amount of traffic of each user,define intent uniIntent: for endpoint('dorms') add middlebox('traffic monitor'),for,add,não
campi_004,University of Illinois - Urbana Champaign,CounterStrike server is blocked by the University firewall,define intent uniIntent: for endpoint('university') add middlebox('firewall') block service('CounterStrike'),for,add | block,não
campi_005,University of Illinois - Urbana Champaign,AIM chat and file transfering is allowed by the University firewall,"define intent uniIntent: for endpoint('university') add middlebox('firewall') allow service('AIM chat'), service('file transfer')",for,add | allow,não
campi_006,University of Illinois - Urbana Champaign,Battlenet is allowed by the University firewall,define intent uniIntent: for endpoint('university') add middlebox('firewall') allow service('Battlenet'),for,add | allow,não
campi_007,University of Illinois - Urbana Champaign,H323 video conferencing is allowed by the University firewall,define intent uniIntent: for endpoint('university') add middlebox('firewall') allow traffic('H323 video conferencing'),for,add | allow,não
campi_008,University of Illinois - Urbana Champaign,Everquest is blocked by the University firewall,define intent uniIntent: for endpoint('university') add middlebox('firewall') block service('Everquest'),for,add | block,não
campi_009,University of Illinois - Urbana Champaign,HTTP and HTTPS are allowed by the University firewall,"define intent uniIntent: for endpoint('university') add middlebox('firewall') allow protocol('HTTP'), protocol('HTTPS')",for,add | allow,não
campi_010,University of Illinois - Urbana Champaign,IMAP and secure IMAP are allowed by the University firewall,"define intent uniIntent: for endpoint('university') add middlebox('firewall') allow protocol('IMAP'), protocol('secure IMAP')",for,add | allow,não


Bloco 2 concluído
F0 localizada em: /kaggle/input/datasets/thiagoarajoguedes/f0-operacional
F1 localizada em: /kaggle/input/datasets/thiagoarajoguedes/f1-operacional
Registros integrados: 50
Status: OK


## Bloco 3 - Preparação das solicitações ao modelo professor

### Objetivo

Este bloco transforma os 50 exemplos e os templates da F0 em seis solicitações independentes ao modelo professor.

### Divisão em lotes

São criados:

- dois lotes de R1;
- dois lotes de R2;
- dois lotes de R3.

Cada lote contém 25 IDs. A divisão reduz o tamanho de cada resposta, diminui o risco de truncamento e facilita a identificação de IDs ausentes ou repetidos.

### Conteúdo de cada tarefa

- **R1:** recebe `id`, entrada em linguagem natural e IR;
- **R2:** recebe `id`, IR e Nile;
- **R3:** recebe `id`, entrada em linguagem natural e Nile.

O professor não cria novas IRs nem novas expressões Nile. Ele explica relações já estabelecidas por F0 e F1.

### Política de idioma

As instruções do prompt permanecem em português. Entretanto:

- o valor de `r1`, `r2` ou `r3` deve estar integralmente em inglês;
- nomes de campos da IR não são traduzidos;
- operadores e palavras-chave Nile não são traduzidos;
- identificadores, entidades, unidades e valores literais são preservados;
- a resposta deve conter somente o JSON solicitado, sem Markdown ou introdução.

### Arquivos esperados

As respostas iniciais devem ser salvas como:

```text
f2_r1_lote_01.json
f2_r1_lote_02.json
f2_r2_lote_01.json
f2_r2_lote_02.json
f2_r3_lote_01.json
f2_r3_lote_02.json
```

### Saídas

O bloco produz:

- seis prompts preenchidos;
- `indice_solicitacoes.csv`;
- `manifest_solicitacoes.json`;
- `f2_solicitacoes_professor.zip`.

### Resultado esperado

Os seis prompts devem conter 25 IDs únicos cada e cobrir os 50 exemplos em cada tipo de justificativa.

In [3]:
# ----------------------------------------------------------
# 3.1 Verificação dos arquivos-base de prompts da F0
# ----------------------------------------------------------

TEMPLATE_PATHS = {
    "R1": F0_DIR / "prompts/f2_geracao_r1.txt",
    "R2": F0_DIR / "prompts/f2_geracao_r2.txt",
    "R3": F0_DIR / "prompts/f2_geracao_r3.txt",
    "REVISAO": F0_DIR / "prompts/f2_revisao_justificativa_reprovada.txt",
}

for chave, caminho in TEMPLATE_PATHS.items():
    if not caminho.is_file():
        raise FileNotFoundError(f"Template obrigatório ausente para {chave}: {caminho}")


# ----------------------------------------------------------
# 3.2 Política de idioma e preservação formal
# ----------------------------------------------------------

INSTRUCAO_IDIOMA_SAIDA = "Escreva cada justificativa integralmente em inglês."

CABECALHO_COMUM_PROMPT = """Você é o modelo professor de um experimento reproduzível de tradução de intenções de rede em linguagem natural para a linguagem formal Nile.

Definições:
- Linguagem natural (NL): enunciado de entrada escrito em inglês.
- Nile: linguagem formal de destino utilizada como resposta esperada do experimento.
- CAMPI: conjunto de referência com 50 pares de NL e Nile.
- Representação Intermediária (IR): estrutura JSON controlada e derivada deterministicamente da estrutura formal da expressão Nile de referência.
- R1: justificativa explícita da passagem de NL para IR.
- R2: justificativa explícita da passagem de IR para Nile.
- R3: justificativa explícita da passagem direta de NL para Nile.

Requisitos de idioma e preservação:
1. As instruções deste prompt estão em português, mas o conteúdo textual de cada justificativa deve ser produzido integralmente em inglês.
2. Escreva cada justificativa integralmente em inglês.
3. Use inglês técnico, conciso e composto por frases completas.
4. Preserve exatamente todos os nomes de campos JSON, campos estruturais, valores controlados, palavras-chave da Nile, operadores, identificadores, nomes de serviços, protocolos, grupos, endpoints e middleboxes, unidades e valores literais fornecidos no exemplo.
5. Não traduza nem parafraseie valores formais.
6. Não altere a entrada NL, a IR ou a referência Nile.
7. Não acrescente informações que não sejam sustentadas pelo exemplo fornecido.
8. Não escreva o conteúdo dos campos de justificativa em português ou em outro idioma diferente do inglês.

Requisitos da saída JSON:
1. Retorne somente JSON válido.
2. Não use Markdown nem cercas de código.
3. Não inclua introduções, comentários, explicações fora do JSON ou texto após o fechamento do objeto.
4. Use aspas duplas em todas as chaves e strings.
5. Não coloque vírgula após o último item de um objeto ou de uma lista.
6. Use true, false e null conforme o padrão JSON, quando necessário.
7. Não inclua campos que não foram solicitados.
"""

TEMPLATE_R1 = CABECALHO_COMUM_PROMPT + """
Tarefa:
Gere uma justificativa R1 para cada exemplo fornecido.

Uma justificativa R1 explica como o enunciado NL em inglês é mapeado para a Representação Intermediária. Ela deve identificar as informações relevantes do enunciado e explicar como essas informações são organizadas nos campos da IR.

Exemplos fornecidos:
{{IR_JSON}}

Requisitos específicos:
1. Gere exatamente uma justificativa R1 para cada exemplo.
2. Preserve o id original de cada exemplo.
3. Mantenha cada justificativa concisa, objetiva e verificável.
4. Mencione scope, source, destination, targets, operations, items e temporal_constraint somente quando esses elementos estiverem presentes na IR.
5. Explique a passagem do enunciado NL para a IR sem inventar fatos intermediários.
6. Retorne somente os campos id e r1.
7. O valor do campo r1 deve estar integralmente em inglês.

Formato obrigatório:
{
  "items": [
    {
      "id": "campi_001",
      "r1": "English rationale explaining the mapping from NL to IR."
    }
  ]
}
"""

TEMPLATE_R2 = CABECALHO_COMUM_PROMPT + """
Tarefa:
Gere uma justificativa R2 para cada exemplo fornecido.

Uma justificativa R2 explica como os campos da Representação Intermediária são compostos na expressão Nile correspondente.

Exemplos fornecidos:
{{IR_JSON}}

Requisitos específicos:
1. Gere exatamente uma justificativa R2 para cada exemplo.
2. Preserve o id original de cada exemplo.
3. Mantenha cada justificativa concisa, objetiva e verificável.
4. Relacione scope, source, destination, targets, operations, items, argumentos e temporal_constraint da IR com a sintaxe Nile correspondente somente quando esses elementos estiverem presentes.
5. Preserve a ordem das operações quando a expressão Nile de referência contiver várias operações.
6. Não acrescente informações ausentes da IR ou da referência Nile.
7. Retorne somente os campos id e r2.
8. O valor do campo r2 deve estar integralmente em inglês.

Formato obrigatório:
{
  "items": [
    {
      "id": "campi_001",
      "r2": "English rationale explaining the mapping from IR to Nile."
    }
  ]
}
"""

TEMPLATE_R3 = CABECALHO_COMUM_PROMPT + """
Tarefa:
Gere uma justificativa R3 para cada exemplo fornecido.

Uma justificativa R3 explica a passagem direta do enunciado NL em inglês para a expressão Nile correspondente. Ela será utilizada somente como artefato demonstrativo em exemplos few-shot.

Exemplos fornecidos:
{{CAMPI_JSON}}

Requisitos específicos:
1. Gere exatamente uma justificativa R3 para cada exemplo.
2. Preserve o id original de cada exemplo.
3. Mantenha cada justificativa concisa, objetiva e verificável.
4. Explique ação, escopo, entidades, relações, argumentos e restrições temporais somente quando esses elementos estiverem presentes no par NL-Nile.
5. Preserve o vocabulário formal e os valores literais da referência Nile.
6. Não acrescente informações ausentes do par fornecido.
7. Retorne somente os campos id e r3.
8. O valor do campo r3 deve estar integralmente em inglês.

Formato obrigatório:
{
  "items": [
    {
      "id": "campi_001",
      "r3": "English rationale explaining the direct mapping from NL to Nile."
    }
  ]
}
"""

TEMPLATE_REVISAO = CABECALHO_COMUM_PROMPT + """
Tarefa:
Corrija a justificativa reprovada.

Tipo de justificativa:
{{TIPO_JUSTIFICATIVA}}

Exemplo:
{{EXEMPLO_JSON}}

Justificativa reprovada:
{{JUSTIFICATIVA_REPROVADA}}

Erros de validação:
{{ERROS_VALIDACAO_JSON}}

Elementos obrigatórios ausentes:
{{ELEMENTOS_FALTANTES_JSON}}

Elementos corretamente representados que devem ser preservados:
{{ELEMENTOS_CORRETOS_JSON}}

Instrução objetiva de correção:
{{INSTRUCAO_CORRECAO}}

Requisitos específicos:
1. Preserve o id e o tipo de justificativa do exemplo.
2. Inclua explicitamente todos os elementos listados como ausentes.
3. Preserve as informações listadas como corretas, sem alterar seus valores formais.
4. Corrija somente os problemas informados pelo validador.
5. Escreva o valor da justificativa corrigida integralmente em inglês.
6. Mantenha a justificativa concisa, objetiva e verificável.
7. Remova afirmações não sustentadas.
8. Não introduza entidades, operadores, relações, valores, unidades ou horários ausentes do exemplo.
9. Preserve exatamente campos formais, palavras-chave da Nile, identificadores, unidades e valores literais fornecidos.
10. Não altere a entrada NL, a IR ou a expressão Nile.
11. Retorne somente o objeto JSON solicitado.

Formato obrigatório:
{
  "id": "{{ID_EXEMPLO}}",
  "tipo": "{{TIPO_JUSTIFICATIVA}}",
  "justificativa": "Corrected rationale written entirely in English."
}
"""

templates_f2 = {
    "R1": TEMPLATE_R1,
    "R2": TEMPLATE_R2,
    "R3": TEMPLATE_R3,
    "REVISAO": TEMPLATE_REVISAO,
}

placeholders_esperados = {
    "R1": ["IR_JSON"],
    "R2": ["IR_JSON"],
    "R3": ["CAMPI_JSON"],
    "REVISAO": [
        "ELEMENTOS_CORRETOS_JSON",
        "ELEMENTOS_FALTANTES_JSON",
        "ERROS_VALIDACAO_JSON",
        "EXEMPLO_JSON",
        "ID_EXEMPLO",
        "INSTRUCAO_CORRECAO",
        "JUSTIFICATIVA_REPROVADA",
        "TIPO_JUSTIFICATIVA",
    ],
}

for chave, template in templates_f2.items():
    observados = identificar_placeholders(template)
    esperados = sorted(placeholders_esperados[chave])
    if observados != esperados:
        raise ValueError(
            f"Placeholders inesperados no template {chave}. "
            f"Esperados: {esperados}. Observados: {observados}."
        )

    texto_normalizado = template.casefold()
    if INSTRUCAO_IDIOMA_SAIDA.casefold() not in texto_normalizado:
        raise ValueError(
            f"O template {chave} não exige que a justificativa seja produzida em inglês."
        )
    if "tarefa:" not in texto_normalizado or "requisitos" not in texto_normalizado:
        raise ValueError(f"O template {chave} não está documentado em português.")


# ----------------------------------------------------------
# 3.3 Preparação dos conteúdos específicos de R1, R2 e R3
# ----------------------------------------------------------

def preparar_exemplo_professor(tipo: str, registro: dict) -> dict:
    if tipo == "R1":
        return {
            "id": registro["id"],
            "nl": registro["nl"],
            "ir": registro["ir"],
        }
    if tipo == "R2":
        return {
            "id": registro["id"],
            "ir": registro["ir"],
            "nile": registro["nile"],
        }
    if tipo == "R3":
        return {
            "id": registro["id"],
            "nl": registro["nl"],
            "nile": registro["nile"],
        }
    raise ValueError(f"Tipo de justificativa desconhecido: {tipo}")


# ----------------------------------------------------------
# 3.4 Construção determinística dos seis lotes
# ----------------------------------------------------------

solicitacoes_professor = []

for tipo in RATIONALE_TYPES:
    exemplos_tipo = [preparar_exemplo_professor(tipo, item) for item in registros_base]

    for indice_lote, inicio in enumerate(range(0, EXPECTED_EXAMPLES, BATCH_SIZE), start=1):
        lote = exemplos_tipo[inicio: inicio + BATCH_SIZE]
        ids_lote = [item["id"] for item in lote]
        request_id = f"f2_{tipo.lower()}_lote_{indice_lote:02d}"
        arquivo_prompt = f"{request_id}_prompt.txt"
        arquivo_resposta = f"{request_id}.json"

        if tipo in {"R1", "R2"}:
            prompt = preencher_template(
                templates_f2[tipo],
                {"IR_JSON": json_legivel({"items": lote})},
            )
        else:
            prompt = preencher_template(
                templates_f2[tipo],
                {"CAMPI_JSON": json_legivel({"items": lote})},
            )

        if INSTRUCAO_IDIOMA_SAIDA not in prompt:
            raise ValueError(f"A política de idioma da justificativa não foi incorporada em {request_id}.")

        caminho_prompt = REQUESTS_DIR / arquivo_prompt
        salvar_texto(prompt, caminho_prompt)

        solicitacoes_professor.append({
            "request_id": request_id,
            "tipo_justificativa": tipo,
            "idioma_prompt": "Português",
            "idioma_justificativa": "English",
            "lote": indice_lote,
            "ids": ids_lote,
            "total_exemplos": len(ids_lote),
            "primeiro_id": ids_lote[0],
            "ultimo_id": ids_lote[-1],
            "arquivo_prompt": arquivo_prompt,
            "arquivo_resposta": arquivo_resposta,
            "caminho_prompt": caminho_prompt,
            "prompt": prompt,
            "caracteres_prompt": len(prompt),
            "sha256_prompt": calcular_sha256_texto(prompt),
        })

if len(solicitacoes_professor) != EXPECTED_GENERATION_BATCHES:
    raise ValueError("A quantidade de solicitações geradas não corresponde ao esperado.")


# ----------------------------------------------------------
# 3.5 Índice e manifesto das solicitações
# ----------------------------------------------------------

indice_solicitacoes_df = pd.DataFrame([
    {
        "request_id": item["request_id"],
        "tipo_justificativa": item["tipo_justificativa"],
        "idioma_prompt": item["idioma_prompt"],
        "idioma_justificativa": item["idioma_justificativa"],
        "lote": item["lote"],
        "total_exemplos": item["total_exemplos"],
        "primeiro_id": item["primeiro_id"],
        "ultimo_id": item["ultimo_id"],
        "arquivo_prompt": item["arquivo_prompt"],
        "arquivo_resposta": item["arquivo_resposta"],
        "caracteres_prompt": item["caracteres_prompt"],
        "sha256_prompt": item["sha256_prompt"],
    }
    for item in solicitacoes_professor
])

indice_solicitacoes_df.to_csv(REQUESTS_INDEX_PATH, index=False, encoding="utf-8")

manifesto_solicitacoes = {
    "fase": FASE,
    "modelo_professor": TEACHER_MODEL,
    "dataset": DATASET_ID,
    "total_exemplos": EXPECTED_EXAMPLES,
    "tipos": list(RATIONALE_TYPES),
    "idioma_prompts": "Português",
    "idioma_justificativas": "English",
    "tamanho_lote": BATCH_SIZE,
    "total_solicitacoes": len(solicitacoes_professor),
    "instrucoes": {
        "responder_apenas_json": True,
        "escrever_justificativas_em_ingles": True,
        "preservar_campos_e_valores_formais": True,
        "preservar_nome_arquivo_resposta": True,
        "adicionar_respostas_como_dataset_kaggle": True,
    },
    "solicitacoes": [
        {
            "request_id": item["request_id"],
            "tipo_justificativa": item["tipo_justificativa"],
            "idioma_prompt": item["idioma_prompt"],
            "idioma_justificativa": item["idioma_justificativa"],
            "lote": item["lote"],
            "ids": item["ids"],
            "arquivo_prompt": item["arquivo_prompt"],
            "arquivo_resposta": item["arquivo_resposta"],
            "sha256_prompt": item["sha256_prompt"],
        }
        for item in solicitacoes_professor
    ],
}

salvar_json(manifesto_solicitacoes, REQUESTS_MANIFEST_PATH)


# ----------------------------------------------------------
# 3.6 Empacotamento das solicitações
# ----------------------------------------------------------

itens_zip_solicitacoes = [
    (item["caminho_prompt"], f"prompts/{item['arquivo_prompt']}")
    for item in solicitacoes_professor
]
itens_zip_solicitacoes.extend([
    (REQUESTS_INDEX_PATH, "indice_solicitacoes.csv"),
    (REQUESTS_MANIFEST_PATH, "manifest_solicitacoes.json"),
])

criar_zip(F2_REQUESTS_ZIP_PATH, itens_zip_solicitacoes)


# ----------------------------------------------------------
# 3.7 Exibição do resumo dos lotes
# ----------------------------------------------------------

exibir_tabela(
    indice_solicitacoes_df,
    titulo="Solicitações preparadas para o modelo professor",
)

resumo_tipos_df = (
    indice_solicitacoes_df
    .groupby(["tipo_justificativa", "idioma_prompt", "idioma_justificativa"], as_index=False)
    .agg(
        lotes=("lote", "count"),
        total_exemplos=("total_exemplos", "sum"),
        caracteres_minimos=("caracteres_prompt", "min"),
        caracteres_maximos=("caracteres_prompt", "max"),
    )
)

exibir_tabela(
    resumo_tipos_df,
    titulo="Resumo das solicitações por tipo de justificativa",
)

print("Bloco 3 concluído")
print(f"Solicitações preparadas: {len(solicitacoes_professor)}")
print("Idioma dos prompts: Português")
print("Idioma obrigatório das justificativas: English")
print(f"Pacote gerado: {F2_REQUESTS_ZIP_PATH}")
print("Status: OK")

ID da solicitação,tipo de justificativa,idioma do prompt,idioma da justificativa,lote,exemplos,primeiro ID,último ID,arquivo do prompt,arquivo esperado da resposta,caracteres do prompt,SHA-256 do prompt
f2_r1_lote_01,R1,Português,English,1,25,campi_001,campi_025,f2_r1_lote_01_prompt.txt,f2_r1_lote_01.json,25163,3ddc3cd07efcb4ab3e168c5a03631c22a61a96bc50be9a005b1d359753bad583
f2_r1_lote_02,R1,Português,English,2,25,campi_026,campi_050,f2_r1_lote_02_prompt.txt,f2_r1_lote_02.json,24052,96ff37519ca894b6c6fadcc5d9bc24499b8c65d4322f65f1d6c5359674ab62d5
f2_r2_lote_01,R2,Português,English,1,25,campi_001,campi_025,f2_r2_lote_01_prompt.txt,f2_r2_lote_01.json,26200,ddf1c09cee5607819d377f6300f4bdf962d9527689890217414f4548e0eca89c
f2_r2_lote_02,R2,Português,English,2,25,campi_026,campi_050,f2_r2_lote_02_prompt.txt,f2_r2_lote_02.json,23935,32bf2600499025d7b05897a530325eb73b4dd5a35bdd6c2e39e014fb04266325
f2_r3_lote_01,R3,Português,English,1,25,campi_001,campi_025,f2_r3_lote_01_prompt.txt,f2_r3_lote_01.json,9287,e0bca5ba01892b63bb1317223b58e921cffb8ba9e0abe035479f0697c16fd40e
f2_r3_lote_02,R3,Português,English,2,25,campi_026,campi_050,f2_r3_lote_02_prompt.txt,f2_r3_lote_02.json,9845,a475068c48b090b41bab7542d4df0fb0859530b6c6c890fea956924494bed7fb


tipo de justificativa,idioma do prompt,idioma da justificativa,lotes,exemplos,caracteres minimos,caracteres maximos
R1,Português,English,2,50,24052,25163
R2,Português,English,2,50,23935,26200
R3,Português,English,2,50,9287,9845


Bloco 3 concluído
Solicitações preparadas: 6
Idioma dos prompts: Português
Idioma obrigatório das justificativas: English
Pacote gerado: /kaggle/working/f2_solicitacoes_professor.zip
Status: OK


## Bloco 4 - Ponto de controle e carga das respostas do modelo professor

### Objetivo

Este bloco impede que a validação seja iniciada antes da disponibilização das seis respostas externas.

### Interação

O notebook pergunta se os arquivos já estão disponíveis:

```text
Os seis arquivos gerados pelo modelo professor já estão disponíveis? [S/N]
```

- `N` encerra esta etapa sem erro e mantém os blocos seguintes bloqueados;
- `S` autoriza a busca, a leitura e a conferência dos arquivos.

A resposta `S` não significa aprovação das justificativas. Ela apenas informa que os arquivos podem ser carregados.

### Localização

As respostas são procuradas recursivamente nos inputs do Kaggle e no diretório configurado. O notebook identifica os arquivos pelos nomes exatos, mesmo quando o dataset é montado em uma pasta adicional.

### Conferências iniciais

O bloco verifica:

- presença dos seis arquivos;
- JSON bem formado;
- chave raiz esperada;
- tipo de justificativa correspondente ao nome;
- 25 itens por lote;
- IDs pertencentes ao intervalo correto;
- ausência de duplicação entre os lotes.

### Estado da execução

Quando os arquivos estão incompletos, o notebook informa quais faltam e não libera a validação. Quando os seis são válidos estruturalmente, a variável de controle permite a execução dos Blocos 5 a 8.

### Resultado esperado

A tabela apresenta os seis arquivos, caminhos localizados, quantidade de itens e status de leitura.

In [4]:
# ----------------------------------------------------------
# 4.1 Confirmação da disponibilidade das respostas iniciais
# ----------------------------------------------------------

ARQUIVOS_PROFESSOR_DECLARADOS_DISPONIVEIS = solicitar_sim_nao(
    "Os seis arquivos gerados pelo modelo professor já estão disponíveis "
    "no dataset do Kaggle? [S/N]: ",
    variavel_ambiente="F2_CONFIRMAR_RESPOSTAS_PROFESSOR",
)

if not ARQUIVOS_PROFESSOR_DECLARADOS_DISPONIVEIS:
    F2_CONTINUAR_VALIDACAO = False
    PROFESSOR_RESPONSES_AVAILABLE = False

    print()
    print("As respostas ainda não foram declaradas como disponíveis.")
    print(f"Pacote de solicitações: {F2_REQUESTS_ZIP_PATH}")
    print("Adicione os seis JSONs ao dataset do Kaggle.")
    print("Depois, execute novamente somente a partir do Bloco 4 e responda S.")
    print("Os Blocos 5 a 11 serão ignorados nesta execução.")
    print("Status: AGUARDANDO RESPOSTAS DO MODELO PROFESSOR")
else:
    # ----------------------------------------------------------
    # 4.1 Definição das raízes de busca das respostas
    # ----------------------------------------------------------

    def preparar_raizes_respostas():
        raizes = []
        caminho_env = os.environ.get("F2_PROFESSOR_RESPONSES_DIR")

        if caminho_env:
            caminho = Path(caminho_env).expanduser().resolve()
            if caminho.is_file() and caminho.suffix.lower() == ".zip":
                extrair_zip(caminho, RESPONSES_EXTRACT_DIR)
                raizes.append(RESPONSES_EXTRACT_DIR)
            elif caminho.is_dir():
                raizes.append(caminho)
            else:
                raise FileNotFoundError(
                    "F2_PROFESSOR_RESPONSES_DIR não aponta para um diretório ou ZIP válido."
                )

        if KAGGLE_INPUT_DIR.exists():
            raizes.append(KAGGLE_INPUT_DIR)
        raizes.append(Path.cwd())

        return [Path(raiz).resolve() for raiz in raizes if Path(raiz).exists()]


    RAIZES_RESPOSTAS = preparar_raizes_respostas()


    # ----------------------------------------------------------
    # 4.2 Localização segura de um arquivo de resposta
    # ----------------------------------------------------------

    def localizar_resposta_professor(nome_arquivo: str):
        candidatos = []
        nome_base = Path(nome_arquivo).stem
        nomes_aceitos = {nome_arquivo, nome_base + ".txt"}

        for raiz in RAIZES_RESPOSTAS:
            for nome in nomes_aceitos:
                for caminho in raiz.rglob(nome):
                    if not caminho.is_file():
                        continue
                    if (
                        caminho_esta_dentro(F2_WORK_DIR, caminho)
                        and not caminho_esta_dentro(RESPONSES_EXTRACT_DIR, caminho)
                    ):
                        continue
                    if caminho_esta_dentro(F2_FINAL_DIR, caminho):
                        continue
                    candidatos.append(caminho.resolve())

        candidatos = sorted(set(candidatos), key=lambda p: (len(str(p)), str(p)))
        if not candidatos:
            return None

        hashes = {calcular_sha256(caminho) for caminho in candidatos}
        if len(hashes) > 1:
            raise RuntimeError(
                f"Foram encontradas respostas diferentes com o nome {nome_arquivo}: "
                + ", ".join(str(caminho) for caminho in candidatos)
            )

        return candidatos[0]


    # ----------------------------------------------------------
    # 4.3 Carga e extração do JSON de cada resposta encontrada
    # ----------------------------------------------------------

    status_respostas = []
    geracoes_professor = []
    respostas_por_request_id = {}

    for solicitacao in solicitacoes_professor:
        caminho_resposta = localizar_resposta_professor(solicitacao["arquivo_resposta"])
        encontrada = caminho_resposta is not None
        resposta_bruta = None
        resposta_json = None
        itens_extraidos = None
        erro_carga = None

        if encontrada:
            resposta_bruta = ler_texto(caminho_resposta)
            try:
                resposta_json = extrair_json_de_resposta(resposta_bruta)
                if not isinstance(resposta_json, dict):
                    raise TypeError("A resposta deve ser um objeto JSON.")
                if not isinstance(resposta_json.get("items"), list):
                    raise TypeError("A resposta deve conter a lista 'items'.")
                itens_extraidos = len(resposta_json["items"])
                respostas_por_request_id[solicitacao["request_id"]] = resposta_json
            except Exception as exc:
                erro_carga = f"{type(exc).__name__}: {exc}"

        status_respostas.append({
            "request_id": solicitacao["request_id"],
            "tipo_justificativa": solicitacao["tipo_justificativa"],
            "lote": solicitacao["lote"],
            "arquivo_resposta": solicitacao["arquivo_resposta"],
            "resposta_encontrada": encontrada,
            "caminho_resposta": None if caminho_resposta is None else str(caminho_resposta),
            "itens_extraidos": itens_extraidos,
            "status": "approved" if encontrada and erro_carga is None else (
                "rejected" if encontrada else "pending"
            ),
            "erros": erro_carga,
        })

        geracoes_professor.append({
            "stage": "initial_generation",
            "request_id": solicitacao["request_id"],
            "rationale_type": solicitacao["tipo_justificativa"],
            "batch": solicitacao["lote"],
            "ids": solicitacao["ids"],
            "teacher_model": TEACHER_MODEL,
            "prompt_file": solicitacao["arquivo_prompt"],
            "prompt_sha256": solicitacao["sha256_prompt"],
            "prompt_text": solicitacao["prompt"],
            "response_file": solicitacao["arquivo_resposta"],
            "response_path": None if caminho_resposta is None else str(caminho_resposta),
            "response_sha256": None if caminho_resposta is None else calcular_sha256(caminho_resposta),
            "response_raw": resposta_bruta,
            "response_loaded": encontrada and erro_carga is None,
            "load_error": erro_carga,
        })

    status_respostas_df = pd.DataFrame(status_respostas)

    respostas_encontradas = int(status_respostas_df["resposta_encontrada"].sum())
    respostas_validamente_carregadas = int((status_respostas_df["status"] == "approved").sum())
    PROFESSOR_RESPONSES_AVAILABLE = (
        respostas_validamente_carregadas == EXPECTED_GENERATION_BATCHES
    )

    if respostas_encontradas > 0 and respostas_validamente_carregadas != respostas_encontradas:
        erros = status_respostas_df.loc[
            status_respostas_df["status"] == "rejected",
            ["arquivo_resposta", "erros"],
        ].to_dict("records")
        raise ValueError(f"Há respostas encontradas, mas não carregáveis: {erros}")


    # ----------------------------------------------------------
    # 4.4 Exibição do estado das respostas
    # ----------------------------------------------------------

    exibir_tabela(
        status_respostas_df,
        titulo="Disponibilidade das respostas do modelo professor",
    )

    print("Bloco 4 concluído")
    print(f"Respostas esperadas: {EXPECTED_GENERATION_BATCHES}")
    print(f"Respostas encontradas: {respostas_encontradas}")
    print(f"Respostas carregadas: {respostas_validamente_carregadas}")

    if PROFESSOR_RESPONSES_AVAILABLE:
        print("Todas as respostas foram localizadas. A validação será executada.")
        print("Status: OK")
    else:
        faltantes = status_respostas_df.loc[
            status_respostas_df["status"] == "pending",
            "arquivo_resposta",
        ].tolist()
        print("A F2 está aguardando respostas do modelo professor.")
        print("Arquivos ainda necessários:")
        for nome in faltantes:
            print(f"- {nome}")
        print(f"Use os prompts contidos em: {F2_REQUESTS_ZIP_PATH}")
        print("Status: AGUARDANDO RESPOSTAS")


    F2_CONTINUAR_VALIDACAO = bool(PROFESSOR_RESPONSES_AVAILABLE)

    if F2_CONTINUAR_VALIDACAO:
        print("Ponto de controle liberado. Os Blocos 5 a 8 podem ser executados.")
    else:
        print("A confirmação foi S, mas os seis arquivos válidos não foram localizados.")
        print("Corrija o dataset e execute novamente somente a partir do Bloco 4.")
        print("Os Blocos 5 a 11 serão ignorados nesta execução.")

Os seis arquivos gerados pelo modelo professor já estão disponíveis no dataset do Kaggle? [S/N]:  S


ID da solicitação,tipo de justificativa,lote,arquivo esperado da resposta,resposta encontrada,caminho da resposta,itens extraídos,status,erros
f2_r1_lote_01,R1,1,f2_r1_lote_01.json,sim,/kaggle/input/datasets/thiagoarajoguedes/f2-respostas-professor/f2_r1_lote_01.json,25,aprovado,-
f2_r1_lote_02,R1,2,f2_r1_lote_02.json,sim,/kaggle/input/datasets/thiagoarajoguedes/f2-respostas-professor/f2_r1_lote_02.json,25,aprovado,-
f2_r2_lote_01,R2,1,f2_r2_lote_01.json,sim,/kaggle/input/datasets/thiagoarajoguedes/f2-respostas-professor/f2_r2_lote_01.json,25,aprovado,-
f2_r2_lote_02,R2,2,f2_r2_lote_02.json,sim,/kaggle/input/datasets/thiagoarajoguedes/f2-respostas-professor/f2_r2_lote_02.json,25,aprovado,-
f2_r3_lote_01,R3,1,f2_r3_lote_01.json,sim,/kaggle/input/datasets/thiagoarajoguedes/f2-respostas-professor/f2_r3_lote_01.json,25,aprovado,-
f2_r3_lote_02,R3,2,f2_r3_lote_02.json,sim,/kaggle/input/datasets/thiagoarajoguedes/f2-respostas-professor/f2_r3_lote_02.json,25,aprovado,-


Bloco 4 concluído
Respostas esperadas: 6
Respostas encontradas: 6
Respostas carregadas: 6
Todas as respostas foram localizadas. A validação será executada.
Status: OK
Ponto de controle liberado. Os Blocos 5 a 8 podem ser executados.


## Bloco 5 - Extração, padronização mecânica e módulo de validação

### Objetivo

Este bloco consolida as 150 respostas iniciais e prepara o módulo determinístico utilizado para avaliar R1, R2 e R3.

### Extração

Para cada item, o bloco recupera:

- ID;
- tipo de justificativa;
- texto produzido;
- arquivo de origem;
- lote;
- resposta bruta associada.

Apenas operações mecânicas são permitidas, como remoção de espaços externos e verificação de tipo. O conteúdo não é reescrito nem traduzido.

### Cópia imutável

As respostas iniciais são preservadas em uma estrutura separada. Essa cópia permite que o Bloco 9 aplique revisões seletivas sem perder o texto originalmente produzido pelo professor e sem executar novamente os blocos anteriores.

### `rationale_core.py`

O módulo implementa verificações comuns, incluindo:

- extensão mínima e máxima;
- idioma inglês;
- rejeição de sinais claros de português;
- ausência de Markdown e metalinguagem;
- operadores e flexões equivalentes;
- presença de valores formais;
- escopos e direções;
- itens, argumentos e horários;
- detecção de valores formais inventados;
- identificação de âncoras esperadas, encontradas e faltantes.

### Testes do módulo

São usados exemplos positivos e negativos para confirmar que:

- uma justificativa adequada é aceita;
- texto em português é rejeitado;
- resposta metalinguística é rejeitada;
- omissões formais são registradas;
- formas verbais equivalentes são reconhecidas.

### Resultado esperado

Devem ser extraídas exatamente 150 justificativas. O módulo é compilado e testado antes das validações específicas.

In [5]:
if not (F2_CONTINUAR_VALIDACAO):
    print("Bloco 5 ignorado porque as respostas iniciais ainda não foram liberadas no Bloco 4.")
    print("Status: NÃO EXECUTADO")
else:
    # ----------------------------------------------------------
    # 5.1 Conteúdo do módulo rationale_core.py
    # ----------------------------------------------------------

    RATIONALE_CORE_CODE = r'''from __future__ import annotations

    import re
    from typing import Any, Dict, List


    MIN_LENGTH = 40
    MAX_LENGTH = 900

    ENGLISH_SIGNALS = {
        "the", "a", "an", "and", "or", "of", "as", "on", "in", "by", "this",
        "that", "which", "to", "from", "for", "with", "without", "is", "are",
        "maps", "mapped", "represents", "represented", "identifies", "identified",
        "renders", "rendered", "translates", "translated", "statement", "intent",
        "expression", "field", "fields", "scope", "operation", "target",
        "source", "destination", "constraint", "becomes", "become", "yield",
        "yields", "produces", "produced", "captures", "captured",
    }

    PORTUGUESE_SIGNALS = {
        "frase", "intenção", "intencao", "escopo", "operação", "operacao",
        "operações", "operacoes", "aplicada", "aplicado", "sobre", "sem",
        "restrição", "restricao", "entrada", "justificativa", "representa",
        "identifica", "traduz", "tradução", "traducao", "expressão", "expressao",
        "estrutura", "configura", "configurado", "adiciona", "bloqueia", "permite",
        "para", "pela", "pelo", "dos", "das",
    }

    FORBIDDEN_PATTERNS = [
        r"as an ai language model",
        r"as a language model",
        r"i cannot",
        r"i can't",
        r"como modelo de linguagem",
        r"não posso",
        r"nao posso",
        r"não tenho acesso",
        r"nao tenho acesso",
        r"chain[ -]?of[ -]?thought",
        r"racioc[ií]nio interno",
        r"system instructions",
        r"instru[cç][õo]es do sistema",
        r"original prompt",
        r"prompt original",
    ]

    OPERATOR_FORMS = {
        "add": (
            "add", "adds", "added", "adding", "insert", "inserts", "inserted",
            "introduce", "introduces", "introduced", "deploy", "deploys", "deployed",
        ),
        "allow": (
            "allow", "allows", "allowed", "allowing", "permit", "permits",
            "permitted", "permitting",
        ),
        "block": (
            "block", "blocks", "blocked", "blocking", "prohibit", "prohibits",
            "prohibited", "prevent", "prevents", "prevented", "deny", "denies",
            "denied",
        ),
        "set": (
            "set", "sets", "setting", "configure", "configures", "configured",
            "configuring", "assign", "assigns", "assigned", "limit", "limits",
            "limited", "cap", "caps", "capped",
        ),
        "unset": (
            "unset", "unsets", "unsetting", "remove", "removes", "removed",
            "removing", "no bandwidth limit", "without bandwidth limit",
            "no rate limit", "without rate limits",
        ),
    }

    R1_CUES = (
        "ir", "intermediate representation", "map", "maps", "mapped", "mapping",
        "represent", "represents", "represented", "capture", "captures", "captured",
        "scope", "operation", "target",
    )

    R2_CUES = (
        "nile", "render", "renders", "rendered", "translate", "translates",
        "translated", "composition", "compose", "composes", "construct",
        "constructs", "constructed", "become", "becomes", "yield", "yields",
        "produce", "produces", "generate", "generates", "definition",
        "starts with", "followed by",
    )

    R2_FORMAL_TERMS = (
        "intent_id", "intent", "scope", "source", "destination", "target",
        "targets", "endpoint", "group", "operation", "operator", "add", "allow",
        "block", "set", "unset", "quota", "bandwidth", "middlebox", "protocol",
        "service", "traffic", "temporal constraint", "start", "end", "for",
        "from", "to",
    )

    R3_CUES = (
        "nl", "natural language", "intent", "map", "maps", "mapped", "translate",
        "translates", "translated", "express", "expressed", "represent",
        "represented", "capture", "captured", "model", "modeled", "realize",
        "realized", "implement", "implemented", "correspond", "corresponding",
        "reflect", "reflecting", "directly",
    )

    FORMAL_CALL_PATTERN = re.compile(
        r"\b(?:endpoint|group|middlebox|protocol|service|traffic|quota|bandwidth|hour)"
        r"\s*\(([^()]*)\)",
        flags=re.IGNORECASE,
    )


    def normalize_text(text: Any) -> str:
        return re.sub(r"\s+", " ", str(text)).strip()


    def _normalized(text: Any) -> str:
        return normalize_text(text).casefold()


    def _mentions(text: str, value: Any) -> bool:
        value_text = normalize_text(value)
        if not value_text:
            return False

        text_norm = _normalized(text)
        value_norm = value_text.casefold()

        if re.fullmatch(r"[A-Za-z0-9_\-]+", value_text):
            return re.search(
                r"(?<![A-Za-z0-9_])" + re.escape(value_norm) + r"(?![A-Za-z0-9_])",
                text_norm,
            ) is not None

        return value_norm in text_norm


    def _mentions_any(text: str, values) -> bool:
        return any(_mentions(text, value) for value in values)


    def _scope_values(ir: Dict[str, Any]) -> List[str]:
        scope = ir["scope"]
        values = []

        source = scope.get("source")
        destination = scope.get("destination")

        if isinstance(source, dict) and source.get("value"):
            values.append(str(source["value"]))
        if isinstance(destination, dict) and destination.get("value"):
            values.append(str(destination["value"]))

        for target in scope.get("targets", []):
            if target.get("value"):
                values.append(str(target["value"]))

        return values


    def _temporal_values(ir: Dict[str, Any]) -> List[str]:
        temporal = ir.get("temporal_constraint")
        if not isinstance(temporal, dict):
            return []
        return [str(temporal["start"]), str(temporal["end"])]


    def _formal_literals(text: str) -> List[str]:
        values = []
        for arguments in FORMAL_CALL_PATTERN.findall(str(text)):
            values.extend(
                normalize_text(item)
                for item in re.findall(r"'([^']*)'", arguments)
            )
        return values


    def _allowed_formal_literals(record: Dict[str, Any]) -> set[str]:
        ir = record["ir"]
        allowed = set()

        scope = ir["scope"]
        for endpoint in (scope.get("source"), scope.get("destination")):
            if isinstance(endpoint, dict):
                for field in ("kind", "value"):
                    if endpoint.get(field) not in (None, ""):
                        allowed.add(_normalized(endpoint[field]))

        for target in scope.get("targets", []):
            for field in ("kind", "value"):
                if target.get(field) not in (None, ""):
                    allowed.add(_normalized(target[field]))

        for operation in ir.get("operations", []):
            if operation.get("operator"):
                allowed.add(_normalized(operation["operator"]))
            for item in operation.get("items", []):
                for field in ("kind", "constraint", "value", "unit"):
                    if item.get(field) not in (None, ""):
                        allowed.add(_normalized(item[field]))

        for value in _temporal_values(ir):
            allowed.add(_normalized(value))

        return allowed


    def _operator_mentioned(text: str, operator: str) -> bool:
        forms = OPERATOR_FORMS.get(operator, (operator,))
        return _mentions_any(text, forms)


    def _semantic_item_mentioned(text: str, item: Dict[str, Any]) -> bool:
        kind = str(item.get("kind", ""))
        value = item.get("value")

        if kind == "traffic" and str(value).casefold() == "any":
            return _mentions(text, "any") or _mentions(text, "all traffic")

        anchor = value if value not in (None, "") else kind
        return _mentions(text, anchor)


    def _rationale_role_mentioned(rationale_type: str, text: str) -> bool:
        if rationale_type == "R1":
            return _mentions_any(text, R1_CUES)

        if rationale_type == "R2":
            if _mentions_any(text, R2_CUES):
                return True
            formal_hits = sum(_mentions(text, term) for term in R2_FORMAL_TERMS)
            return formal_hits >= 2

        return _mentions_any(text, R3_CUES)


    def validate_common(text: Any) -> Dict[str, Any]:
        errors = []
        warnings = []

        if not isinstance(text, str):
            return {
                "approved": False,
                "errors": ["A justificativa deve ser uma string."],
                "warnings": [],
                "characters": 0,
            }

        clean = normalize_text(text)
        length = len(clean)

        if not clean:
            errors.append("A justificativa está vazia.")
        if length < MIN_LENGTH:
            errors.append(f"A justificativa possui menos de {MIN_LENGTH} caracteres.")
        if length > MAX_LENGTH:
            errors.append(f"A justificativa possui mais de {MAX_LENGTH} caracteres.")
        if "```" in text:
            errors.append("A justificativa contém bloco de código Markdown.")
        if clean.startswith("{") or clean.startswith("["):
            errors.append("A justificativa contém estrutura JSON no campo textual.")

        clean_norm = clean.casefold()
        for pattern in FORBIDDEN_PATTERNS:
            if re.search(pattern, clean_norm, flags=re.IGNORECASE):
                errors.append("A justificativa contém resposta metalinguística ou raciocínio oculto.")
                break

        tokens = set(re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ_]+", clean_norm))
        english_hits = tokens.intersection(ENGLISH_SIGNALS)
        portuguese_hits = tokens.intersection(PORTUGUESE_SIGNALS)
        possui_acento_portugues = bool(re.search(r"[áàâãéêíóôõúç]", clean_norm))

        if len(english_hits) < 2:
            errors.append("A justificativa não contém evidência suficiente de prosa em inglês.")
        if possui_acento_portugues or len(portuguese_hits) >= 2:
            errors.append("A justificativa deve ser escrita integralmente em inglês.")

        if length > 650:
            warnings.append("A justificativa é longa para uso como artefato demonstrativo.")

        return {
            "approved": not errors,
            "errors": errors,
            "warnings": warnings,
            "characters": length,
        }


    def _validate_grounding(
        rationale_type: str,
        record: Dict[str, Any],
        text: str,
    ) -> Dict[str, Any]:
        errors = []
        warnings = []
        ir = record["ir"]

        expected_anchors = []
        matched_anchors = []

        def register(anchor: str, matched: bool, error_message: str) -> None:
            expected_anchors.append(anchor)
            if matched:
                matched_anchors.append(anchor)
            else:
                errors.append(error_message)

        for operation in ir.get("operations", []):
            operator = str(operation.get("operator", ""))
            register(
                f"operator:{operator}",
                _operator_mentioned(text, operator),
                f"O operator '{operator}' não foi representado na justificativa.",
            )

        for value in _scope_values(ir):
            register(
                f"scope:{value}",
                _mentions(text, value),
                f"O elemento de scope '{value}' não foi mencionado.",
            )

        scope_type = ir["scope"]["type"]
        if scope_type in {"route", "route_for"}:
            direction_ok = (
                _mentions_any(text, ("from", "source", "origin"))
                and _mentions_any(text, ("to", "destination"))
            )
            register(
                "scope:route_direction",
                direction_ok,
                "A direção source/from para destination/to não foi representada.",
            )

        for operation_index, operation in enumerate(ir.get("operations", [])):
            for item_index, item in enumerate(operation.get("items", [])):
                kind = str(item.get("kind", ""))

                if rationale_type == "R2":
                    register(
                        f"operation:{operation_index}:item:{item_index}:kind:{kind}",
                        _mentions(text, kind),
                        f"O kind '{kind}' da operation não foi mencionado em R2.",
                    )

                    for field in ("constraint", "value", "unit"):
                        value = item.get(field)
                        if value in (None, ""):
                            continue
                        register(
                            f"operation:{operation_index}:item:{item_index}:{field}:{value}",
                            _mentions(text, value),
                            f"O campo '{field}' com valor '{value}' não foi mencionado em R2.",
                        )
                else:
                    anchor = item.get("value") or item.get("kind")
                    register(
                        f"operation:{operation_index}:item:{item_index}:{anchor}",
                        _semantic_item_mentioned(text, item),
                        f"O item '{anchor}' da operation não foi representado.",
                    )

        for value in _temporal_values(ir):
            register(
                f"temporal:{value}",
                _mentions(text, value),
                f"O valor temporal '{value}' não foi mencionado.",
            )

        register(
            f"role:{rationale_type}",
            _rationale_role_mentioned(rationale_type, text),
            f"A justificativa não apresenta sinais suficientes do papel de {rationale_type}.",
        )

        allowed_literals = _allowed_formal_literals(record)
        unsupported_literals = [
            value
            for value in _formal_literals(text)
            if _normalized(value) not in allowed_literals
        ]
        if unsupported_literals:
            errors.append(
                "Foram encontrados valores formais não sustentados pelo exemplo: "
                + ", ".join(unsupported_literals)
            )

        unique_expected = list(dict.fromkeys(expected_anchors))
        unique_matched = list(dict.fromkeys(matched_anchors))
        coverage = (
            len(unique_matched) / len(unique_expected)
            if unique_expected
            else 1.0
        )

        if coverage < 0.75:
            warnings.append("A cobertura das âncoras verificáveis ficou abaixo de 75%.")

        return {
            "approved": not errors,
            "errors": errors,
            "warnings": warnings,
            "expected_anchors": unique_expected,
            "matched_anchors": unique_matched,
            "anchor_coverage": coverage,
        }


    def validate_rationale(
        rationale_type: str,
        record: Dict[str, Any],
        text: Any,
    ) -> Dict[str, Any]:
        rationale_type = str(rationale_type).upper()
        if rationale_type not in {"R1", "R2", "R3"}:
            raise ValueError(f"Tipo de justificativa inválido: {rationale_type}")

        common = validate_common(text)
        clean = normalize_text(text)

        grounding = _validate_grounding(rationale_type, record, clean) if clean else {
            "approved": False,
            "errors": [],
            "warnings": [],
            "expected_anchors": [],
            "matched_anchors": [],
            "anchor_coverage": 0.0,
        }

        errors = common["errors"] + grounding["errors"]
        warnings = common["warnings"] + grounding["warnings"]

        return {
            "approved": len(errors) == 0,
            "normalized_text": clean,
            "errors": errors,
            "warnings": warnings,
            "characters": common["characters"],
            "expected_anchors": grounding["expected_anchors"],
            "matched_anchors": grounding["matched_anchors"],
            "anchor_coverage": grounding["anchor_coverage"],
        }

    '''

    RATIONALE_CORE_CODE = "\n".join(
        linha[4:] if linha.startswith("    ") else linha
        for linha in RATIONALE_CORE_CODE.splitlines()
    )
    salvar_texto(RATIONALE_CORE_CODE, RATIONALE_CORE_PATH)
    py_compile.compile(str(RATIONALE_CORE_PATH), doraise=True)
    rationale_core = carregar_modulo_python(RATIONALE_CORE_PATH, "f2_rationale_core")

    # Remove somente caches automáticos; os módulos-fonte permanecem preservados.
    for diretorio_cache in F2_WORK_DIR.rglob("__pycache__"):
        shutil.rmtree(diretorio_cache, ignore_errors=True)
    for arquivo_pyc in F2_WORK_DIR.rglob("*.pyc"):
        arquivo_pyc.unlink(missing_ok=True)


    # ----------------------------------------------------------
    # 5.2 Testes determinísticos do módulo de validação
    # ----------------------------------------------------------

    registro_r1_teste = next(item for item in registros_base if item["id"] == "campi_001")
    registro_r2_teste = next(item for item in registros_base if item["id"] == "campi_036")

    texto_r1_valido_teste = (
        "The statement identifies a for scope targeting students and maps it to an "
        "add operation with copyright monitoring in the IR."
    )

    texto_r2_valido_teste = (
        "The intent definition starts with define intent uniIntent. The for scope "
        "targets group('students'). The set operation produces "
        "quota('download', '5', 'gb/d'), followed by start hour('06:00') "
        "and end hour('05:59')."
    )

    texto_r2_sem_scope_teste = (
        "The set operation produces quota('download', '5', 'gb/d'), followed by "
        "start hour('06:00') and end hour('05:59')."
    )

    texto_literal_invalido_teste = (
        "The IR maps the statement to an add operation for group('invented') "
        "with middlebox('copyright monitoring')."
    )

    texto_portugues_teste = (
        "A entrada identifica o escopo for para students e representa a operação add "
        "com copyright monitoring na IR."
    )

    texto_metalinguistico_teste = (
        "As a language model, I cannot provide this rationale."
    )

    resultado_r1_valido_teste = rationale_core.validate_rationale(
        "R1", registro_r1_teste, texto_r1_valido_teste
    )
    resultado_r2_valido_teste = rationale_core.validate_rationale(
        "R2", registro_r2_teste, texto_r2_valido_teste
    )
    resultado_r2_sem_scope_teste = rationale_core.validate_rationale(
        "R2", registro_r2_teste, texto_r2_sem_scope_teste
    )
    resultado_literal_invalido_teste = rationale_core.validate_rationale(
        "R1", registro_r1_teste, texto_literal_invalido_teste
    )
    resultado_portugues_teste = rationale_core.validate_rationale(
        "R1", registro_r1_teste, texto_portugues_teste
    )
    resultado_metalinguistico_teste = rationale_core.validate_rationale(
        "R1", registro_r1_teste, texto_metalinguistico_teste
    )

    if not resultado_r1_valido_teste["approved"]:
        raise AssertionError(
            "O caso positivo de R1 foi rejeitado: "
            + str(resultado_r1_valido_teste["errors"])
        )
    if not resultado_r2_valido_teste["approved"]:
        raise AssertionError(
            "O caso positivo de R2 foi rejeitado: "
            + str(resultado_r2_valido_teste["errors"])
        )
    if resultado_r2_sem_scope_teste["approved"]:
        raise AssertionError("O caso R2 sem o valor do scope foi aceito indevidamente.")
    if resultado_literal_invalido_teste["approved"]:
        raise AssertionError("O caso com literal formal inventado foi aceito indevidamente.")
    if resultado_portugues_teste["approved"]:
        raise AssertionError("O caso em português foi aceito indevidamente.")
    if resultado_metalinguistico_teste["approved"]:
        raise AssertionError("O caso metalinguístico foi aceito indevidamente.")


    # ----------------------------------------------------------
    # 5.3 Extração e normalização dos seis lotes
    # ----------------------------------------------------------

    candidatos_por_id = {
        item["id"]: {"id": item["id"], "r1": None, "r2": None, "r3": None}
        for item in registros_base
    }

    registros_base_por_id = {item["id"]: item for item in registros_base}
    erros_extracao = []
    resumo_extracao = []

    if PROFESSOR_RESPONSES_AVAILABLE:
        for solicitacao in solicitacoes_professor:
            request_id = solicitacao["request_id"]
            tipo = solicitacao["tipo_justificativa"]
            chave_texto = tipo.lower()
            resposta = respostas_por_request_id[request_id]
            itens = resposta["items"]
            ids_esperados = solicitacao["ids"]

            if len(itens) != len(ids_esperados):
                erros_extracao.append(
                    f"{request_id}: esperados {len(ids_esperados)} itens; recebidos {len(itens)}."
                )

            ids_observados = []
            campos_esperados = {"id", chave_texto}

            for indice, item in enumerate(itens, start=1):
                if not isinstance(item, dict):
                    erros_extracao.append(f"{request_id}, item {indice}: não é objeto JSON.")
                    continue

                campos_observados = set(item)
                if campos_observados != campos_esperados:
                    erros_extracao.append(
                        f"{request_id}, item {indice}: campos esperados {sorted(campos_esperados)}; "
                        f"observados {sorted(campos_observados)}."
                    )

                item_id = str(item.get("id", "")).strip()
                texto = item.get(chave_texto)
                ids_observados.append(item_id)

                if item_id not in ids_esperados:
                    erros_extracao.append(f"{request_id}: ID inesperado: {item_id}")
                    continue
                if not isinstance(texto, str):
                    erros_extracao.append(
                        f"{request_id}, {item_id}: o campo {chave_texto} deve ser string."
                    )
                    continue

                candidatos_por_id[item_id][chave_texto] = rationale_core.normalize_text(texto)

            if len(ids_observados) != len(set(ids_observados)):
                erros_extracao.append(f"{request_id}: existem IDs duplicados.")
            if set(ids_observados) != set(ids_esperados):
                ausentes = sorted(set(ids_esperados) - set(ids_observados))
                extras = sorted(set(ids_observados) - set(ids_esperados))
                erros_extracao.append(
                    f"{request_id}: IDs ausentes={ausentes}; IDs extras={extras}."
                )

            resumo_extracao.append({
                "request_id": request_id,
                "tipo_justificativa": tipo,
                "lote": solicitacao["lote"],
                "total_exemplos": len(ids_esperados),
                "itens_extraidos": len(itens),
                "status": "approved",
            })

        if erros_extracao:
            raise ValueError("Falhas na extração das justificativas:\n- " + "\n- ".join(erros_extracao))

        for item_id, registro in candidatos_por_id.items():
            for chave in ("r1", "r2", "r3"):
                if not registro[chave]:
                    raise ValueError(f"Justificativa {chave.upper()} ausente para {item_id}.")




    # Cópia imutável das justificativas iniciais.
    # O Bloco 9 restaura esta base antes de aplicar qualquer revisão.
    candidatos_iniciais_por_id = copy.deepcopy(candidatos_por_id)


    # ----------------------------------------------------------
    # 5.4 Exibição do resumo de extração
    # ----------------------------------------------------------

    if PROFESSOR_RESPONSES_AVAILABLE:
        exibir_tabela(
            pd.DataFrame(resumo_extracao),
            titulo="Extração dos lotes de justificativas",
        )

        tamanho_candidatos_df = pd.DataFrame([
            {
                "id": item_id,
                "r1_caracteres": len(registro["r1"]),
                "r2_caracteres": len(registro["r2"]),
                "r3_caracteres": len(registro["r3"]),
            }
            for item_id, registro in sorted(candidatos_por_id.items())
        ])

        exibir_tabela(
            tamanho_candidatos_df,
            titulo="Tamanho das justificativas extraídas",
            altura_px=480,
        )

        print("Justificativas extraídas: 150")
        print("Status: OK")
    else:
        print("O módulo de validação foi criado e testado.")
        print("A extração foi adiada porque as respostas iniciais ainda não estão completas.")
        print("Status: AGUARDANDO RESPOSTAS")

    print("Bloco 5 concluído")

ID da solicitação,tipo de justificativa,lote,exemplos,itens extraídos,status
f2_r1_lote_01,R1,1,25,25,aprovado
f2_r1_lote_02,R1,2,25,25,aprovado
f2_r2_lote_01,R2,1,25,25,aprovado
f2_r2_lote_02,R2,2,25,25,aprovado
f2_r3_lote_01,R3,1,25,25,aprovado
f2_r3_lote_02,R3,2,25,25,aprovado


ID,caracteres de R1,caracteres de R2,caracteres de R3
campi_001,310,319,180
campi_002,214,290,161
campi_003,181,292,184
campi_004,210,371,189
campi_005,219,390,167
campi_006,177,338,154
campi_007,211,371,188
campi_008,184,338,184
campi_009,192,369,154
campi_010,206,381,179


Justificativas extraídas: 150
Status: OK
Bloco 5 concluído


## Bloco 6 - Validação das justificativas R1

### Objetivo

Este bloco verifica se cada R1 explica adequadamente a passagem da entrada em linguagem natural para a IR correspondente.

### Elementos esperados

Conforme a estrutura de cada exemplo, a R1 deve mencionar evidências de:

- escopo da intenção;
- `source`, `destination` ou `targets`;
- operadores;
- valores principais dos itens;
- unidades e constraints;
- restrição temporal;
- relação entre a intenção e os campos estruturais da IR.

A justificativa não precisa copiar o JSON, mas deve tornar explícitas as decisões estruturais relevantes.

### Flexibilidade controlada

O validador reconhece formas linguísticas equivalentes, como:

```text
add, adds, adding
block, blocks, blocking
set, sets, setting
```

Também aceita menções textuais controladas aos valores formais, desde que não introduzam entidades inexistentes.

### Resultado por exemplo

Para cada ID, são registrados:

- aprovação;
- erros;
- âncoras esperadas;
- âncoras encontradas;
- âncoras faltantes;
- idioma e extensão;
- origem da resposta.

### Fluxo de reprovação

Uma reprovação não encerra o notebook. O resultado é enviado ao Bloco 9, que criará um prompt específico somente para o caso afetado.

### Resultado esperado

A tabela apresenta as 50 R1 e o resumo de aprovadas e reprovadas.

In [6]:
if not (F2_CONTINUAR_VALIDACAO):
    print("Bloco 6 ignorado porque a validação ainda não foi liberada no Bloco 4.")
    print("Status: NÃO EXECUTADO")
else:
    # ----------------------------------------------------------
    # 6.1 Validação item a item de R1
    # ----------------------------------------------------------

    def validar_tipo_justificativa(tipo: str):
        resultados = []
        if not PROFESSOR_RESPONSES_AVAILABLE:
            return resultados

        chave = tipo.lower()
        for item_id in sorted(candidatos_por_id):
            texto = candidatos_por_id[item_id][chave]
            resultado = rationale_core.validate_rationale(
                tipo,
                registros_base_por_id[item_id],
                texto,
            )
            resultados.append({
                "id": item_id,
                "tipo_justificativa": tipo,
                "aprovada": resultado["approved"],
                "quantidade_erros": len(resultado["errors"]),
                "quantidade_avisos": len(resultado["warnings"]),
                "erros": " | ".join(resultado["errors"]),
                "avisos": " | ".join(resultado["warnings"]),
                "ancoras_encontradas": len(resultado["matched_anchors"]),
                "ancoras_esperadas": len(resultado["expected_anchors"]),
                "cobertura_ancoras": round(float(resultado["anchor_coverage"]), 4),
                "ancoras_esperadas_lista": list(resultado["expected_anchors"]),
                "ancoras_encontradas_lista": list(resultado["matched_anchors"]),
                "ancoras_faltantes_lista": [
                    ancora
                    for ancora in resultado["expected_anchors"]
                    if ancora not in set(resultado["matched_anchors"])
                ],
                "caracteres": resultado["characters"],
                "texto_normalizado": resultado["normalized_text"],
            })
        return resultados


    validacao_r1 = validar_tipo_justificativa("R1")


    # ----------------------------------------------------------
    # 6.2 Exibição dos resultados de R1
    # ----------------------------------------------------------

    if validacao_r1:
        validacao_r1_df = pd.DataFrame(validacao_r1)
        resumo_r1_df = pd.DataFrame([{
            "tipo_justificativa": "R1",
            "total_justificativas": len(validacao_r1_df),
            "aprovadas": int(validacao_r1_df["aprovada"].sum()),
            "reprovadas": int((~validacao_r1_df["aprovada"]).sum()),
            "status": "approved" if validacao_r1_df["aprovada"].all() else "rejected",
        }])

        exibir_tabela(resumo_r1_df, titulo="Resumo da validação de R1")
        exibir_tabela(
            validacao_r1_df.drop(
                columns=[
                    "texto_normalizado",
                    "ancoras_esperadas_lista",
                    "ancoras_encontradas_lista",
                ],
                errors="ignore",
            ),
            titulo="Validação das 50 justificativas R1",
            altura_px=520,
        )
    else:
        validacao_r1_df = pd.DataFrame()
        print("Validação R1 adiada: respostas iniciais ainda não disponíveis.")

    print("Bloco 6 concluído")

tipo de justificativa,justificativas,aprovadas,reprovadas,status
R1,50,50,0,aprovado


ID,tipo de justificativa,aprovada,quantidade de erros,quantidade de avisos,erros,avisos,âncoras encontradas,âncoras esperadas,cobertura de âncoras,ancoras faltantes lista,caracteres
campi_001,R1,sim,0,0,,,4,4,1.0,[],310
campi_002,R1,sim,0,0,,,4,4,1.0,[],214
campi_003,R1,sim,0,0,,,4,4,1.0,[],181
campi_004,R1,sim,0,0,,,6,6,1.0,[],210
campi_005,R1,sim,0,0,,,7,7,1.0,[],219
campi_006,R1,sim,0,0,,,6,6,1.0,[],177
campi_007,R1,sim,0,0,,,6,6,1.0,[],211
campi_008,R1,sim,0,0,,,6,6,1.0,[],184
campi_009,R1,sim,0,0,,,7,7,1.0,[],192
campi_010,R1,sim,0,0,,,7,7,1.0,[],206


Bloco 6 concluído


## Bloco 7 - Validação das justificativas R2

### Objetivo

Este bloco verifica se cada R2 explica como os campos da IR são compostos na expressão Nile de referência.

### Elementos esperados

Quando presentes, a R2 deve cobrir:

- tipo de escopo;
- `source` e `destination`;
- targets;
- direção de `route` ou `route_for`;
- operadores;
- `kind` dos itens;
- argumentos formais;
- valores e unidades;
- `temporal_constraint`;
- relação de composição ou renderização formal.

A palavra `Nile` não é obrigatória quando o texto descreve inequivocamente a construção da expressão por meio de seus componentes.

### Verificação completa da composição

R2 recebe critérios mais detalhados do que R1 e R3 porque deve explicar a passagem entre duas representações formais. O validador procura não apenas o valor principal, mas também kinds e argumentos que distinguem itens estruturalmente diferentes.

### Exemplo de omissão

Uma justificativa que descreva corretamente cota, valor, unidade e horário, mas não mencione o target ao qual a política se aplica, pode ser classificada como incompleta.

### Resultado por exemplo

São registrados aprovação, erros, âncoras formais e componentes ausentes. Esses dados alimentam diretamente o feedback estruturado do Bloco 9.

### Resultado esperado

A tabela apresenta as 50 R2 e identifica somente os casos que exigem revisão seletiva.

In [7]:
if not (F2_CONTINUAR_VALIDACAO):
    print("Bloco 7 ignorado porque a validação ainda não foi liberada no Bloco 4.")
    print("Status: NÃO EXECUTADO")
else:
    # ----------------------------------------------------------
    # 7.1 Validação item a item de R2
    # ----------------------------------------------------------

    validacao_r2 = validar_tipo_justificativa("R2")


    # ----------------------------------------------------------
    # 7.2 Exibição dos resultados de R2
    # ----------------------------------------------------------

    if validacao_r2:
        validacao_r2_df = pd.DataFrame(validacao_r2)
        resumo_r2_df = pd.DataFrame([{
            "tipo_justificativa": "R2",
            "total_justificativas": len(validacao_r2_df),
            "aprovadas": int(validacao_r2_df["aprovada"].sum()),
            "reprovadas": int((~validacao_r2_df["aprovada"]).sum()),
            "status": "approved" if validacao_r2_df["aprovada"].all() else "rejected",
        }])

        exibir_tabela(resumo_r2_df, titulo="Resumo da validação de R2")
        exibir_tabela(
            validacao_r2_df.drop(
                columns=[
                    "texto_normalizado",
                    "ancoras_esperadas_lista",
                    "ancoras_encontradas_lista",
                ],
                errors="ignore",
            ),
            titulo="Validação das 50 justificativas R2",
            altura_px=520,
        )
    else:
        validacao_r2_df = pd.DataFrame()
        print("Validação R2 adiada: respostas iniciais ainda não disponíveis.")

    print("Bloco 7 concluído")

tipo de justificativa,justificativas,aprovadas,reprovadas,status
R2,50,49,1,reprovado


ID,tipo de justificativa,aprovada,quantidade de erros,quantidade de avisos,erros,avisos,âncoras encontradas,âncoras esperadas,cobertura de âncoras,ancoras faltantes lista,caracteres
campi_001,R2,sim,0,0,,,5,5,1.0000,[],319
campi_002,R2,sim,0,0,,,4,4,1.0000,[],290
campi_003,R2,sim,0,0,,,5,5,1.0000,[],292
campi_004,R2,sim,0,0,,,8,8,1.0000,[],371
campi_005,R2,sim,0,0,,,10,10,1.0000,[],390
campi_006,R2,sim,0,0,,,8,8,1.0000,[],338
campi_007,R2,sim,0,0,,,8,8,1.0000,[],371
campi_008,R2,sim,0,0,,,8,8,1.0000,[],338
campi_009,R2,sim,0,0,,,10,10,1.0000,[],369
campi_010,R2,sim,0,0,,,10,10,1.0000,[],381


Bloco 7 concluído


## Bloco 8 - Validação das justificativas R3

### Objetivo

Este bloco verifica se cada R3 explica diretamente a tradução da intenção em linguagem natural para Nile.

### Elementos esperados

Conforme o exemplo, a R3 deve mencionar:

- escopo e entidades afetadas;
- direção em escopos de rota;
- operadores;
- valores principais dos itens;
- unidades ou constraints;
- horários;
- relação direta entre o requisito textual e a política formal.

A IR não precisa ser mencionada, pois R3 representa o caminho direto NL → Nile.

### Equivalências controladas

O validador pode reconhecer formulações semanticamente equivalentes quando elas são explicitamente previstas, como `all traffic` para `traffic('any')`. Essa flexibilidade não autoriza paráfrases irrestritas nem valores inventados.

### Critério de aprovação

A justificativa deve ser curta, em inglês e suficientemente específica para explicar a referência. Textos genéricos, metalinguísticos ou que apenas afirmam que a tradução está correta são rejeitados.

### Resultado por exemplo

Assim como nos blocos anteriores, são preservados:

- erros;
- âncoras esperadas;
- âncoras encontradas;
- âncoras faltantes;
- status final.

### Resultado esperado

As 50 R3 são consolidadas e encaminhadas ao Bloco 9 junto com R1 e R2.

In [8]:
if not (F2_CONTINUAR_VALIDACAO):
    print("Bloco 8 ignorado porque a validação ainda não foi liberada no Bloco 4.")
    print("Status: NÃO EXECUTADO")
else:
    # ----------------------------------------------------------
    # 8.1 Validação item a item de R3
    # ----------------------------------------------------------

    validacao_r3 = validar_tipo_justificativa("R3")


    # ----------------------------------------------------------
    # 8.2 Exibição dos resultados de R3
    # ----------------------------------------------------------

    if validacao_r3:
        validacao_r3_df = pd.DataFrame(validacao_r3)
        resumo_r3_df = pd.DataFrame([{
            "tipo_justificativa": "R3",
            "total_justificativas": len(validacao_r3_df),
            "aprovadas": int(validacao_r3_df["aprovada"].sum()),
            "reprovadas": int((~validacao_r3_df["aprovada"]).sum()),
            "status": "approved" if validacao_r3_df["aprovada"].all() else "rejected",
        }])

        exibir_tabela(resumo_r3_df, titulo="Resumo da validação de R3")
        exibir_tabela(
            validacao_r3_df.drop(
                columns=[
                    "texto_normalizado",
                    "ancoras_esperadas_lista",
                    "ancoras_encontradas_lista",
                ],
                errors="ignore",
            ),
            titulo="Validação das 50 justificativas R3",
            altura_px=520,
        )
    else:
        validacao_r3_df = pd.DataFrame()
        print("Validação R3 adiada: respostas iniciais ainda não disponíveis.")

    print("Bloco 8 concluído")

tipo de justificativa,justificativas,aprovadas,reprovadas,status
R3,50,50,0,aprovado


ID,tipo de justificativa,aprovada,quantidade de erros,quantidade de avisos,erros,avisos,âncoras encontradas,âncoras esperadas,cobertura de âncoras,ancoras faltantes lista,caracteres
campi_001,R3,sim,0,0,,,4,4,1.0,[],180
campi_002,R3,sim,0,0,,,4,4,1.0,[],161
campi_003,R3,sim,0,0,,,4,4,1.0,[],184
campi_004,R3,sim,0,0,,,6,6,1.0,[],189
campi_005,R3,sim,0,0,,,7,7,1.0,[],167
campi_006,R3,sim,0,0,,,6,6,1.0,[],154
campi_007,R3,sim,0,0,,,6,6,1.0,[],188
campi_008,R3,sim,0,0,,,6,6,1.0,[],184
campi_009,R3,sim,0,0,,,7,7,1.0,[],154
campi_010,R3,sim,0,0,,,7,7,1.0,[],179


Bloco 8 concluído


## Bloco 9 - Revisão seletiva e revalidação

### Objetivo

Este bloco consolida as 150 validações e corrige somente as justificativas que não atenderam aos critérios.

### Quando não há reprovações

Se todas as justificativas forem aprovadas, o bloco:

- mantém as respostas iniciais como versões finais;
- registra que nenhuma revisão foi necessária;
- libera a auditoria automática.

### Quando há reprovações

Para cada caso, o notebook gera um prompt individual contendo:

- tipo da justificativa;
- ID do exemplo;
- entrada, IR e Nile relevantes;
- justificativa anterior;
- erros identificados;
- âncoras faltantes;
- elementos corretos a preservar;
- instrução específica de correção;
- formato JSON esperado.

O feedback é preenchido por placeholders genéricos. Não existe regra exclusiva para um ID.

### Pacote de revisão

`f2_revisoes_pendentes.zip` contém:

- um prompt por caso;
- `indice_revisoes.csv`;
- `feedback_revisoes.jsonl`.

### Segundo ponto de controle

O notebook pergunta se as respostas de revisão estão disponíveis:

- `N` encerra a etapa sem erro;
- `S` localiza os JSONs individuais, restaura a cópia imutável das respostas iniciais, aplica somente as revisões correspondentes e executa novamente a validação.

### Rastreabilidade

A versão inicial não é apagada. O histórico registra qual texto foi substituído, o motivo, o arquivo de revisão e o resultado posterior.

### Resultado esperado

O bloco só libera a finalização quando todas as 150 justificativas estão aprovadas.

In [9]:
if not (F2_CONTINUAR_VALIDACAO):
    print("Bloco 9 ignorado porque as respostas iniciais ainda não foram validadas.")
    print("Status: NÃO EXECUTADO")
else:
    # ----------------------------------------------------------
    # 9.1 Consolidação das validações iniciais
    # ----------------------------------------------------------

    # Restaura a base inicial antes de procurar e aplicar revisões.
    # Isso evita duplicações quando somente o Bloco 9 é executado novamente.
    candidatos_por_id = copy.deepcopy(candidatos_iniciais_por_id)
    geracoes_professor = [
        item for item in geracoes_professor
        if item.get("stage") == "initial_generation"
    ]

    validacoes_iniciais = validacao_r1 + validacao_r2 + validacao_r3
    validacao_inicial_por_chave = {
        (item["id"], item["tipo_justificativa"]): item
        for item in validacoes_iniciais
    }

    reprovacoes_iniciais = [
        item for item in validacoes_iniciais
        if not item["aprovada"]
    ]

    revisoes_aplicadas = []
    status_revisoes = []


    # ----------------------------------------------------------
    # 9.2 Preparação do exemplo e descrição das âncoras
    # ----------------------------------------------------------

    def preparar_exemplo_revisao(tipo: str, registro: dict) -> dict:
        if tipo == "R1":
            return {"id": registro["id"], "nl": registro["nl"], "ir": registro["ir"]}
        if tipo == "R2":
            return {"id": registro["id"], "ir": registro["ir"], "nile": registro["nile"]}
        if tipo == "R3":
            return {"id": registro["id"], "nl": registro["nl"], "nile": registro["nile"]}
        raise ValueError(tipo)


    def localizar_entidade_scope(registro: dict, valor: str):
        scope = registro["ir"]["scope"]
        entidades = []

        for papel in ("source", "destination"):
            entidade = scope.get(papel)
            if isinstance(entidade, dict):
                entidades.append((papel, entidade))

        for entidade in scope.get("targets", []):
            entidades.append(("target", entidade))

        for papel, entidade in entidades:
            if str(entidade.get("value", "")) == str(valor):
                return papel, entidade

        return None, None


    def descrever_ancora(ancora: str, registro: dict) -> str:
        partes = str(ancora).split(":")
        categoria = partes[0] if partes else ""

        if categoria == "operator" and len(partes) >= 2:
            return f"operator '{':'.join(partes[1:])}'"

        if categoria == "scope" and len(partes) >= 2:
            valor = ":".join(partes[1:])
            if valor == "route_direction":
                return "direção explícita de source/from para destination/to"

            papel, entidade = localizar_entidade_scope(registro, valor)
            if entidade is not None:
                chamada = f"{entidade.get('kind')}('{entidade.get('value')}')"
                if papel in {"source", "destination"}:
                    return f"{papel} {chamada} no scope"
                return f"{chamada} no campo scope"
            return f"valor de scope '{valor}'"

        if categoria == "operation" and len(partes) >= 6:
            indice_operacao = partes[1]
            indice_item = partes[3]
            campo = partes[4]
            valor = ":".join(partes[5:])
            return (
                f"campo '{campo}' com valor '{valor}' no item {indice_item} "
                f"da operation {indice_operacao}"
            )

        if categoria == "operation" and len(partes) >= 5:
            indice_operacao = partes[1]
            indice_item = partes[3]
            valor = ":".join(partes[4:])
            return f"item '{valor}' na operation {indice_operacao}, posição {indice_item}"

        if categoria == "temporal" and len(partes) >= 2:
            return f"valor temporal '{':'.join(partes[1:])}'"

        if categoria == "role" and len(partes) >= 2:
            tipo = ":".join(partes[1:])
            descricoes = {
                "R1": "explicação explícita da passagem de NL para IR",
                "R2": "explicação explícita da composição de IR para Nile",
                "R3": "explicação explícita da passagem direta de NL para Nile",
            }
            return descricoes.get(tipo, f"papel textual de {tipo}")

        return str(ancora)


    def separar_mensagens(texto: str):
        return [
            trecho.strip()
            for trecho in str(texto or "").split(" | ")
            if trecho.strip()
        ]


    # ----------------------------------------------------------
    # 9.3 Construção determinística do feedback de revisão
    # ----------------------------------------------------------

    def construir_feedback_revisao(item: dict, registro: dict) -> dict:
        esperadas = list(item.get("ancoras_esperadas_lista", []))
        encontradas = list(item.get("ancoras_encontradas_lista", []))
        conjunto_encontradas = set(encontradas)

        faltantes = [
            ancora for ancora in esperadas
            if ancora not in conjunto_encontradas
        ]
        corretas = [
            ancora for ancora in esperadas
            if ancora in conjunto_encontradas
        ]

        elementos_faltantes = [
            {
                "ancora": ancora,
                "descricao": descrever_ancora(ancora, registro),
            }
            for ancora in faltantes
        ]

        elementos_corretos = [
            {
                "ancora": ancora,
                "descricao": descrever_ancora(ancora, registro),
            }
            for ancora in corretas
            if not str(ancora).startswith("role:")
        ]

        erros = separar_mensagens(item.get("erros", ""))

        if elementos_faltantes:
            descricoes_faltantes = "; ".join(
                elemento["descricao"]
                for elemento in elementos_faltantes
            )
            instrucao = (
                "Reescreva a justificativa em inglês e inclua explicitamente "
                f"os seguintes elementos: {descricoes_faltantes}. "
                "Preserve todas as informações corretas e não acrescente dados "
                "que não estejam no exemplo."
            )
        else:
            instrucao = (
                "Reescreva a justificativa em inglês, corrija todos os erros "
                "listados e preserve as informações sustentadas pelo exemplo."
            )

        return {
            "motivo_reprovacao": (
                "A justificativa não atendeu integralmente aos critérios "
                "determinísticos de validação."
            ),
            "erros_validacao": erros,
            "elementos_faltantes": elementos_faltantes,
            "elementos_corretos_a_preservar": elementos_corretos,
            "instrucao_correcao": instrucao,
        }


    def preencher_prompt_revisao(item: dict, texto_atual: str):
        item_id = item["id"]
        tipo = item["tipo_justificativa"]
        registro = registros_base_por_id[item_id]
        feedback = construir_feedback_revisao(item, registro)

        prompt = preencher_template(
            templates_f2["REVISAO"],
            {
                "TIPO_JUSTIFICATIVA": tipo,
                "ID_EXEMPLO": item_id,
                "EXEMPLO_JSON": json_legivel(
                    preparar_exemplo_revisao(tipo, registro)
                ),
                "JUSTIFICATIVA_REPROVADA": texto_atual,
                "ERROS_VALIDACAO_JSON": json_legivel(
                    feedback["erros_validacao"]
                ),
                "ELEMENTOS_FALTANTES_JSON": json_legivel(
                    feedback["elementos_faltantes"]
                ),
                "ELEMENTOS_CORRETOS_JSON": json_legivel(
                    feedback["elementos_corretos_a_preservar"]
                ),
                "INSTRUCAO_CORRECAO": feedback["instrucao_correcao"],
            },
        )

        placeholders_restantes = identificar_placeholders(prompt)
        if placeholders_restantes:
            raise ValueError(
                "O prompt de revisão contém placeholders não preenchidos: "
                f"{placeholders_restantes}"
            )

        for elemento in feedback["elementos_faltantes"]:
            if elemento["descricao"] not in prompt:
                raise ValueError(
                    "O prompt de revisão não contém a descrição de um elemento faltante: "
                    f"{elemento['descricao']}"
                )

        return prompt, feedback


    # ----------------------------------------------------------
    # 9.4 Preparação dos prompts das reprovações iniciais
    # ----------------------------------------------------------

    solicitacoes_revisao = []

    for item in reprovacoes_iniciais:
        item_id = item["id"]
        tipo = item["tipo_justificativa"]
        chave = tipo.lower()
        arquivo_prompt = f"f2_revisao_{chave}_{item_id}_prompt.txt"
        arquivo_resposta = f"f2_revisao_{chave}_{item_id}.json"

        prompt_revisao, feedback_revisao = preencher_prompt_revisao(
            item,
            candidatos_por_id[item_id][chave],
        )

        caminho_prompt = REVISION_REQUESTS_DIR / arquivo_prompt
        salvar_texto(prompt_revisao, caminho_prompt)

        solicitacoes_revisao.append({
            "request_id": f"revision_{chave}_{item_id}",
            "tipo_justificativa": tipo,
            "id": item_id,
            "arquivo_prompt": arquivo_prompt,
            "arquivo_resposta": arquivo_resposta,
            "caminho_prompt": caminho_prompt,
            "prompt": prompt_revisao,
            "sha256_prompt": calcular_sha256_texto(prompt_revisao),
            "erros_iniciais": item["erros"],
            "feedback": feedback_revisao,
        })




    # ----------------------------------------------------------
    # 9.5 Exportação das solicitações e ponto de controle
    # ----------------------------------------------------------

    def exportar_pacote_revisoes(solicitacoes: list[dict]) -> None:
        registros_indice = []
        registros_feedback = []
        itens_zip = []

        for solicitacao in solicitacoes:
            feedback = solicitacao["feedback"]
            registros_indice.append({
                "id": solicitacao["id"],
                "tipo_justificativa": solicitacao["tipo_justificativa"],
                "arquivo_prompt": solicitacao["arquivo_prompt"],
                "arquivo_resposta": solicitacao["arquivo_resposta"],
                "elementos_faltantes": " | ".join(
                    elemento["descricao"]
                    for elemento in feedback["elementos_faltantes"]
                ),
                "elementos_corretos_a_preservar": " | ".join(
                    elemento["descricao"]
                    for elemento in feedback["elementos_corretos_a_preservar"]
                ),
                "erros": solicitacao["erros_iniciais"],
                "sha256_prompt": solicitacao["sha256_prompt"],
            })
            registros_feedback.append({
                "id": solicitacao["id"],
                "tipo_justificativa": solicitacao["tipo_justificativa"],
                "arquivo_prompt": solicitacao["arquivo_prompt"],
                "arquivo_resposta": solicitacao["arquivo_resposta"],
                "sha256_prompt": solicitacao["sha256_prompt"],
                "feedback": feedback,
            })
            itens_zip.append(
                (
                    solicitacao["caminho_prompt"],
                    f"prompts/{solicitacao['arquivo_prompt']}",
                )
            )

        pd.DataFrame(registros_indice).to_csv(
            REVISION_INDEX_PATH,
            index=False,
            encoding="utf-8",
        )
        salvar_jsonl(registros_feedback, REVISION_FEEDBACK_PATH)

        itens_zip.extend([
            (REVISION_INDEX_PATH, "indice_revisoes.csv"),
            (REVISION_FEEDBACK_PATH, "feedback_revisoes.jsonl"),
        ])
        criar_zip(F2_REVISIONS_ZIP_PATH, itens_zip)


    if solicitacoes_revisao:
        exportar_pacote_revisoes(solicitacoes_revisao)

        REVISOES_PROFESSOR_DECLARADAS_DISPONIVEIS = solicitar_sim_nao(
            "As respostas de revisão do modelo professor já estão disponíveis "
            "no dataset do Kaggle? [S/N]: ",
            variavel_ambiente="F2_CONFIRMAR_REVISOES_PROFESSOR",
        )

        if REVISOES_PROFESSOR_DECLARADAS_DISPONIVEIS:
            arquivos_revisao_faltantes = [
                solicitacao["arquivo_resposta"]
                for solicitacao in solicitacoes_revisao
                if localizar_resposta_professor(
                    solicitacao["arquivo_resposta"]
                ) is None
            ]

            if arquivos_revisao_faltantes:
                F2_CONTINUAR_FINALIZACAO = False
                print()
                print("A confirmação foi S, mas ainda faltam respostas de revisão:")
                for nome in arquivos_revisao_faltantes:
                    print(f"- {nome}")
                print("Atualize o dataset e execute novamente somente o Bloco 9.")
            else:
                F2_CONTINUAR_FINALIZACAO = True
                print("Todas as respostas de revisão foram localizadas.")
        else:
            F2_CONTINUAR_FINALIZACAO = False
            print()
            print(f"Pacote de revisão: {F2_REVISIONS_ZIP_PATH}")
            print("Envie os prompts ao modelo professor e adicione os JSONs ao dataset.")
            print("Depois, execute novamente somente a partir do Bloco 9 e responda S.")
            print("Os Blocos 10 e 11 serão ignorados nesta execução.")
    else:
        REVISOES_PROFESSOR_DECLARADAS_DISPONIVEIS = True
        F2_CONTINUAR_FINALIZACAO = True
        print("Nenhuma revisão seletiva foi necessária.")


    # ----------------------------------------------------------
    # 9.6 Aplicação das revisões, revalidação e consolidação
    # ----------------------------------------------------------

    if F2_CONTINUAR_FINALIZACAO:
        # ----------------------------------------------------------
        # 9.5 Localização e aplicação das respostas de revisão
        # ----------------------------------------------------------

        for solicitacao in solicitacoes_revisao:
            caminho_resposta = localizar_resposta_professor(solicitacao["arquivo_resposta"])
            encontrada = caminho_resposta is not None
            aplicada = False
            erro = None
            resposta_bruta = None

            if encontrada:
                try:
                    resposta_bruta = ler_texto(caminho_resposta)
                    resposta_json = extrair_json_de_resposta(resposta_bruta)

                    campos_esperados = {"id", "tipo", "justificativa"}
                    if not isinstance(resposta_json, dict):
                        raise TypeError("A revisão deve ser um objeto JSON.")
                    if set(resposta_json) != campos_esperados:
                        raise ValueError(
                            f"Campos esperados: {sorted(campos_esperados)}; "
                            f"observados: {sorted(resposta_json)}."
                        )
                    if str(resposta_json["id"]).strip() != solicitacao["id"]:
                        raise ValueError("O ID da revisão não corresponde ao exemplo solicitado.")
                    if str(resposta_json["tipo"]).strip().upper() != solicitacao["tipo_justificativa"]:
                        raise ValueError("O tipo da revisão não corresponde ao tipo solicitado.")
                    if not isinstance(resposta_json["justificativa"], str):
                        raise TypeError("O campo 'justificativa' deve ser string.")

                    chave = solicitacao["tipo_justificativa"].lower()
                    candidatos_por_id[solicitacao["id"]][chave] = rationale_core.normalize_text(
                        resposta_json["justificativa"]
                    )
                    aplicada = True

                    geracoes_professor.append({
                        "stage": "revision",
                        "request_id": solicitacao["request_id"],
                        "rationale_type": solicitacao["tipo_justificativa"],
                        "batch": None,
                        "ids": [solicitacao["id"]],
                        "teacher_model": TEACHER_MODEL,
                        "prompt_file": solicitacao["arquivo_prompt"],
                        "prompt_sha256": solicitacao["sha256_prompt"],
                        "prompt_text": solicitacao["prompt"],
                        "validation_feedback": solicitacao["feedback"],
                        "response_file": solicitacao["arquivo_resposta"],
                        "response_path": str(caminho_resposta),
                        "response_sha256": calcular_sha256(caminho_resposta),
                        "response_raw": resposta_bruta,
                        "response_loaded": True,
                        "load_error": None,
                    })

                    revisoes_aplicadas.append((solicitacao["id"], solicitacao["tipo_justificativa"]))
                except Exception as exc:
                    erro = f"{type(exc).__name__}: {exc}"

            status_revisoes.append({
                "id": solicitacao["id"],
                "tipo_justificativa": solicitacao["tipo_justificativa"],
                "arquivo_resposta": solicitacao["arquivo_resposta"],
                "revisao_necessaria": True,
                "revisao_encontrada": encontrada,
                "revisao_aplicada": aplicada,
                "elementos_faltantes": " | ".join(
                    elemento["descricao"]
                    for elemento in solicitacao["feedback"]["elementos_faltantes"]
                ),
                "erros": erro,
                "status": "approved" if aplicada else "pending",
            })


        # ----------------------------------------------------------
        # 9.6 Revalidação integral após as revisões disponíveis
        # ----------------------------------------------------------

        validacoes_finais = []

        if PROFESSOR_RESPONSES_AVAILABLE:
            for tipo in RATIONALE_TYPES:
                chave = tipo.lower()
                for item_id in sorted(candidatos_por_id):
                    resultado = rationale_core.validate_rationale(
                        tipo,
                        registros_base_por_id[item_id],
                        candidatos_por_id[item_id][chave],
                    )
                    inicial = validacao_inicial_por_chave[(item_id, tipo)]
                    faltantes = [
                        ancora
                        for ancora in resultado["expected_anchors"]
                        if ancora not in set(resultado["matched_anchors"])
                    ]
                    validacoes_finais.append({
                        "id": item_id,
                        "tipo_justificativa": tipo,
                        "aprovada_inicialmente": inicial["aprovada"],
                        "revisao_aplicada": (item_id, tipo) in revisoes_aplicadas,
                        "aprovada_finalmente": resultado["approved"],
                        "quantidade_erros": len(resultado["errors"]),
                        "quantidade_avisos": len(resultado["warnings"]),
                        "erros": " | ".join(resultado["errors"]),
                        "avisos": " | ".join(resultado["warnings"]),
                        "ancoras_encontradas": len(resultado["matched_anchors"]),
                        "ancoras_esperadas": len(resultado["expected_anchors"]),
                        "cobertura_ancoras": round(float(resultado["anchor_coverage"]), 4),
                        "ancoras_esperadas_lista": list(resultado["expected_anchors"]),
                        "ancoras_encontradas_lista": list(resultado["matched_anchors"]),
                        "ancoras_faltantes_lista": faltantes,
                        "elementos_faltantes": " | ".join(
                            descrever_ancora(ancora, registros_base_por_id[item_id])
                            for ancora in faltantes
                        ),
                        "caracteres": resultado["characters"],
                    })

        validacoes_finais_df = pd.DataFrame(validacoes_finais)

        F2_READY = (
            PROFESSOR_RESPONSES_AVAILABLE
            and len(validacoes_finais_df) == EXPECTED_RATIONALES
            and bool(validacoes_finais_df["aprovada_finalmente"].all())
        )


        # ----------------------------------------------------------
        # 9.7 Pacote de revisões pendentes com feedback estruturado
        # ----------------------------------------------------------

        reprovacoes_finais = []
        if not validacoes_finais_df.empty:
            reprovacoes_finais = validacoes_finais_df.loc[
                ~validacoes_finais_df["aprovada_finalmente"]
            ].to_dict("records")

        if reprovacoes_finais:
            revisoes_pendentes = []

            for item in reprovacoes_finais:
                item_id = item["id"]
                tipo = item["tipo_justificativa"]
                chave = tipo.lower()
                arquivo_prompt = f"f2_revisao_{chave}_{item_id}_prompt.txt"
                arquivo_resposta = f"f2_revisao_{chave}_{item_id}.json"

                prompt_atualizado, feedback_atualizado = preencher_prompt_revisao(
                    item,
                    candidatos_por_id[item_id][chave],
                )
                caminho_prompt = REVISION_REQUESTS_DIR / arquivo_prompt
                salvar_texto(prompt_atualizado, caminho_prompt)

                revisoes_pendentes.append({
                    "id": item_id,
                    "tipo_justificativa": tipo,
                    "arquivo_prompt": arquivo_prompt,
                    "arquivo_resposta": arquivo_resposta,
                    "caminho_prompt": caminho_prompt,
                    "sha256_prompt": calcular_sha256_texto(prompt_atualizado),
                    "erros": item["erros"],
                    "feedback": feedback_atualizado,
                })

            indice_revisoes_df = pd.DataFrame([
                {
                    "id": item["id"],
                    "tipo_justificativa": item["tipo_justificativa"],
                    "arquivo_prompt": item["arquivo_prompt"],
                    "arquivo_resposta": item["arquivo_resposta"],
                    "elementos_faltantes": " | ".join(
                        elemento["descricao"]
                        for elemento in item["feedback"]["elementos_faltantes"]
                    ),
                    "elementos_corretos_a_preservar": " | ".join(
                        elemento["descricao"]
                        for elemento in item["feedback"]["elementos_corretos_a_preservar"]
                    ),
                    "erros": item["erros"],
                    "sha256_prompt": item["sha256_prompt"],
                }
                for item in revisoes_pendentes
            ])
            indice_revisoes_df.to_csv(REVISION_INDEX_PATH, index=False, encoding="utf-8")

            feedback_revisoes = [
                {
                    "id": item["id"],
                    "tipo_justificativa": item["tipo_justificativa"],
                    "arquivo_prompt": item["arquivo_prompt"],
                    "arquivo_resposta": item["arquivo_resposta"],
                    "sha256_prompt": item["sha256_prompt"],
                    "feedback": item["feedback"],
                }
                for item in revisoes_pendentes
            ]
            salvar_jsonl(feedback_revisoes, REVISION_FEEDBACK_PATH)

            itens_zip_revisoes = [
                (item["caminho_prompt"], f"prompts/{item['arquivo_prompt']}")
                for item in revisoes_pendentes
            ]
            itens_zip_revisoes.extend([
                (REVISION_INDEX_PATH, "indice_revisoes.csv"),
                (REVISION_FEEDBACK_PATH, "feedback_revisoes.jsonl"),
            ])
            criar_zip(F2_REVISIONS_ZIP_PATH, itens_zip_revisoes)
        else:
            indice_revisoes_df = pd.DataFrame()


        # ----------------------------------------------------------
        # 9.8 Consolidação das justificativas aprovadas
        # ----------------------------------------------------------

        rationales_references = []
        if F2_READY:
            rationales_references = [
                {
                    "id": item_id,
                    "r1": candidatos_por_id[item_id]["r1"],
                    "r2": candidatos_por_id[item_id]["r2"],
                    "r3": candidatos_por_id[item_id]["r3"],
                }
                for item_id in sorted(candidatos_por_id)
            ]

            if len(rationales_references) != EXPECTED_EXAMPLES:
                raise ValueError("A consolidação final não contém 50 registros.")


        # ----------------------------------------------------------
        # 9.9 Exibição do resumo final de validação
        # ----------------------------------------------------------

        if not validacoes_finais_df.empty:
            resumo_final_validacao_df = (
                validacoes_finais_df
                .groupby("tipo_justificativa", as_index=False)
                .agg(
                    total_justificativas=("id", "count"),
                    aprovadas_inicialmente=("aprovada_inicialmente", "sum"),
                    revisoes_aplicadas=("revisao_aplicada", "sum"),
                    aprovadas_finalmente=("aprovada_finalmente", "sum"),
                )
            )
            resumo_final_validacao_df["reprovadas"] = (
                resumo_final_validacao_df["total_justificativas"]
                - resumo_final_validacao_df["aprovadas_finalmente"]
            )
            resumo_final_validacao_df["status"] = np.where(
                resumo_final_validacao_df["reprovadas"] == 0,
                "approved",
                "rejected",
            )

            exibir_tabela(
                resumo_final_validacao_df,
                titulo="Resultado consolidado da validação das justificativas",
            )

            if reprovacoes_finais:
                colunas_reprovacoes = [
                    "id",
                    "tipo_justificativa",
                    "elementos_faltantes",
                    "erros",
                    "cobertura_ancoras",
                ]
                exibir_tabela(
                    validacoes_finais_df.loc[
                        ~validacoes_finais_df["aprovada_finalmente"],
                        colunas_reprovacoes,
                    ],
                    titulo="Justificativas que ainda exigem revisão",
                    altura_px=480,
                )

            if status_revisoes:
                exibir_tabela(
                    pd.DataFrame(status_revisoes),
                    titulo="Situação das respostas de revisão",
                    altura_px=480,
                )

        print("Bloco 9 concluído")
        print(f"Justificativas esperadas: {EXPECTED_RATIONALES}")
        print(f"Revisões aplicadas: {len(revisoes_aplicadas)}")

        if F2_READY:
            print("Todas as justificativas foram aprovadas.")
            print("Status: PRONTO PARA EMPACOTAMENTO")
        elif PROFESSOR_RESPONSES_AVAILABLE and reprovacoes_finais:
            print(f"Justificativas ainda reprovadas: {len(reprovacoes_finais)}")
            print(f"Pacote de revisões: {F2_REVISIONS_ZIP_PATH}")
            print("Status: AGUARDANDO REVISÕES")
        else:
            print("A consolidação aguarda as respostas iniciais do modelo professor.")
            print("Status: AGUARDANDO RESPOSTAS")

    else:
        F2_READY = False
        validacoes_finais_df = pd.DataFrame()
        reprovacoes_finais = reprovacoes_iniciais
        rationales_references = []

        print("Bloco 9 interrompido no ponto de controle das revisões.")
        print("Status: AGUARDANDO RESPOSTAS DE REVISÃO")

As respostas de revisão do modelo professor já estão disponíveis no dataset do Kaggle? [S/N]:  S


Todas as respostas de revisão foram localizadas.


tipo de justificativa,justificativas,aprovadas inicialmente,revisões aplicadas,aprovadas finalmente,reprovadas,status
R1,50,50,0,50,0,aprovado
R2,50,49,1,50,0,aprovado
R3,50,50,0,50,0,aprovado


ID,tipo de justificativa,arquivo esperado da resposta,revisão necessária,revisão encontrada,revisão aplicada,elementos faltantes,erros,status
campi_036,R2,f2_revisao_r2_campi_036.json,sim,sim,sim,group('students') no campo scope,-,aprovado


Bloco 9 concluído
Justificativas esperadas: 150
Revisões aplicadas: 1
Todas as justificativas foram aprovadas.
Status: PRONTO PARA EMPACOTAMENTO


## Bloco 10 - Auditoria final automática

### Objetivo

Este bloco realiza a barreira determinística final da F2. Ele não solicita confirmação manual e não declara revisão qualitativa humana.

### Condições de execução

A auditoria é iniciada somente quando:

- as seis respostas iniciais foram carregadas;
- todas as revisões necessárias foram aplicadas;
- R1, R2 e R3 foram revalidadas;
- não existem justificativas pendentes.

### Base de auditoria

Para cada um dos 50 IDs, o arquivo reúne:

- entrada em linguagem natural;
- IR de referência;
- Nile de referência;
- R1 consolidada;
- R2 consolidada;
- R3 consolidada;
- status final de cada tipo.

### Conferências

O bloco verifica:

- exatamente 50 exemplos;
- IDs únicos;
- exatamente 150 justificativas;
- 50 R1, 50 R2 e 50 R3;
- todas as aprovações verdadeiras;
- ausência de registros incompletos.

### Artefato produzido

A auditoria é salva em:

```text
f2_auditoria_final.csv
```

Esse arquivo é um artefato auxiliar de conferência da execução e não integra os cinco arquivos do pacote operacional.

### Resultado esperado

Quando as 150 justificativas estão aprovadas, o Bloco 11 é liberado automaticamente.

In [10]:
# ----------------------------------------------------------
# 10.1 Preparação da auditoria final automática
# ----------------------------------------------------------

F2_AUDITORIA_APROVADA = False
F2_AUDITORIA_CONCLUIDA_EM_UTC = None
F2_AUDITORIA_PATH = BASE_DIR / "f2_auditoria_final.csv"

if not (
    F2_CONTINUAR_VALIDACAO
    and F2_CONTINUAR_FINALIZACAO
    and F2_READY
):
    print(
        "Bloco 10 ignorado porque a F2 ainda possui "
        "respostas ou revisões pendentes."
    )
    print("Status: NÃO EXECUTADO")
else:
    auditoria_registros = []

    for item_id in sorted(candidatos_por_id):
        base = registros_base_por_id[item_id]

        validacoes_id = validacoes_finais_df.loc[
            validacoes_finais_df["id"] == item_id
        ].set_index("tipo_justificativa")

        tipos_encontrados = set(validacoes_id.index.tolist())
        tipos_esperados = set(RATIONALE_TYPES)

        if tipos_encontrados != tipos_esperados:
            raise ValueError(
                f"Validações incompletas para {item_id}. "
                f"Esperado: {sorted(tipos_esperados)}. "
                f"Encontrado: {sorted(tipos_encontrados)}."
            )

        auditoria_registros.append({
            "id": item_id,
            "nl": base["nl"],
            "ir": json.dumps(
                base["ir"],
                ensure_ascii=False,
                sort_keys=True,
                separators=(",", ":"),
            ),
            "nile": base["nile"],
            "r1": candidatos_por_id[item_id]["r1"],
            "r1_aprovada": bool(
                validacoes_id.loc["R1", "aprovada_finalmente"]
            ),
            "r2": candidatos_por_id[item_id]["r2"],
            "r2_aprovada": bool(
                validacoes_id.loc["R2", "aprovada_finalmente"]
            ),
            "r3": candidatos_por_id[item_id]["r3"],
            "r3_aprovada": bool(
                validacoes_id.loc["R3", "aprovada_finalmente"]
            ),
        })

    auditoria_df = pd.DataFrame(auditoria_registros)

    # ----------------------------------------------------------
    # 10.2 Conferências determinísticas da auditoria
    # ----------------------------------------------------------

    if len(auditoria_df) != EXPECTED_EXAMPLES:
        raise ValueError(
            "A auditoria final não contém exatamente "
            f"{EXPECTED_EXAMPLES} exemplos."
        )

    if auditoria_df["id"].duplicated().any():
        ids_duplicados = sorted(
            auditoria_df.loc[
                auditoria_df["id"].duplicated(keep=False),
                "id",
            ].unique()
        )

        raise ValueError(
            "A auditoria final contém IDs duplicados: "
            + ", ".join(ids_duplicados)
        )

    colunas_aprovacao = [
        "r1_aprovada",
        "r2_aprovada",
        "r3_aprovada",
    ]

    total_justificativas_auditadas = int(
        auditoria_df[colunas_aprovacao].size
    )

    total_justificativas_aprovadas = int(
        auditoria_df[colunas_aprovacao]
        .astype(bool)
        .sum()
        .sum()
    )

    if total_justificativas_auditadas != EXPECTED_RATIONALES:
        raise ValueError(
            "A auditoria final não contém exatamente "
            f"{EXPECTED_RATIONALES} justificativas."
        )

    if total_justificativas_aprovadas != EXPECTED_RATIONALES:
        raise ValueError(
            "A auditoria final encontrou justificativas "
            "ainda não aprovadas."
        )

    if not auditoria_df[colunas_aprovacao].astype(bool).all().all():
        raise ValueError(
            "Uma ou mais justificativas não passaram "
            "pela validação final."
        )

    auditoria_df["status_final"] = "aprovada"

    auditoria_df.to_csv(
        F2_AUDITORIA_PATH,
        index=False,
        encoding="utf-8",
    )

    if not F2_AUDITORIA_PATH.is_file():
        raise FileNotFoundError(
            "O arquivo de auditoria final não foi criado."
        )

    exibir_tabela(
        auditoria_df,
        titulo=(
            "Auditoria final automática das "
            "justificativas consolidadas"
        ),
        altura_px=620,
    )

    F2_AUDITORIA_APROVADA = True
    F2_AUDITORIA_CONCLUIDA_EM_UTC = datetime.now(
        timezone.utc
    ).isoformat()

    print(f"Arquivo de auditoria: {F2_AUDITORIA_PATH}")
    print(f"Exemplos auditados: {len(auditoria_df)}")
    print(
        "Justificativas auditadas e aprovadas: "
        f"{total_justificativas_aprovadas}/"
        f"{EXPECTED_RATIONALES}"
    )
    print(
        "O Bloco 11 está liberado automaticamente "
        "para empacotamento."
    )
    print("Status: OK")

print("Bloco 10 concluído")

ID,entrada em linguagem natural,ir,Nile,r1,r1 aprovada,r2,r2 aprovada,r3,r3 aprovada,status final
campi_001,"If a student is in obvious violations of copyright law by using a room's wired connection (ie ResNet) to distribute copyrighted materials, the room's connection will be disabled, and the issue could be sent to Housing Student Judicial Affairs","{""intent_id"":""uniIntent"",""operations"":[{""items"":[{""kind"":""middlebox"",""value"":""copyright monitoring""}],""operator"":""add""}],""scope"":{""destination"":null,""source"":null,""targets"":[{""kind"":""group"",""value"":""students""}],""type"":""for""},""temporal_constraint"":null}",define intent uniIntent: for group('students') add middlebox('copyright monitoring'),"The NL states that if a student uses a wired connection to violate copyright, the connection will be disabled and the case referred. The IR maps the intent to a 'for' scope targeting the group 'students' and adds a 'copyright monitoring' middlebox, without representing the conditional or the disabling action.",sim,The intent definition starts with 'define intent uniIntent:'. The scope type 'for' and the target of kind 'group' with value 'students' produce 'for group('students')'. The operation with operator 'add' and an item of kind 'middlebox' and value 'copyright monitoring' results in 'add middlebox('copyright monitoring')'.,sim,"The intent adds a copyright monitoring middlebox for the group of students to detect copyright violations from their wired connections, as described in the natural language policy.",sim,aprovada
campi_002,"Currently, the University of Illinois does not have any rate limits","{""intent_id"":""uniIntent"",""operations"":[{""items"":[{""kind"":""bandwidth""}],""operator"":""unset""}],""scope"":{""destination"":null,""source"":null,""targets"":[{""kind"":""endpoint"",""value"":""university""}],""type"":""for""},""temporal_constraint"":null}",define intent uniIntent: for endpoint('university') unset bandwidth(),"The NL asserts that the University of Illinois has no rate limits. The IR reflects this by setting a 'for' scope on the endpoint 'university' and applying an 'unset' operation on bandwidth, removing any rate limit.",sim,The intent definition starts with 'define intent uniIntent:'. The scope type 'for' and the target of kind 'endpoint' with value 'university' yield 'for endpoint('university')'. The operation 'unset' with an item of kind 'bandwidth' without additional parameters becomes 'unset bandwidth()'.,sim,"The intent removes any bandwidth rate limits for the university endpoint, reflecting the statement that the University of Illinois does not have any rate limits.",sim,aprovada
campi_003,University Housing monitors only the amount of traffic of each user,"{""intent_id"":""uniIntent"",""operations"":[{""items"":[{""kind"":""middlebox"",""value"":""traffic monitor""}],""operator"":""add""}],""scope"":{""destination"":null,""source"":null,""targets"":[{""kind"":""endpoint"",""value"":""dorms""}],""type"":""for""},""temporal_constraint"":null}",define intent uniIntent: for endpoint('dorms') add middlebox('traffic monitor'),The NL indicates that University Housing monitors only the traffic volume per user. The IR maps this to a 'for' scope on the endpoint 'dorms' and adds a 'traffic monitor' middlebox.,sim,The intent definition starts with 'define intent uniIntent:'. The scope type 'for' and the target of kind 'endpoint' with value 'dorms' produce 'for endpoint('dorms')'. The operation 'add' with an item of kind 'middlebox' and value 'traffic monitor' yields 'add middlebox('traffic monitor')'.,sim,"The intent adds a traffic monitor middlebox for the dorms endpoint to monitor the amount of traffic per user, matching the description of University Housing monitoring traffic amounts.",sim,aprovada
campi_004,CounterStrike server is blocked by the University firewall,"{""intent_id"":""uniIntent"",""operations"":[{""items"":[{""kind"":""middlebox"",""value"":""firewall""}],""opera

Arquivo de auditoria: /kaggle/working/f2_auditoria_final.csv
Exemplos auditados: 50
Justificativas auditadas e aprovadas: 150/150
O Bloco 11 está liberado automaticamente para empacotamento.
Status: OK
Bloco 10 concluído


## Bloco 11 - Manifesto e empacotamento final

### Objetivo

Este bloco encerra a F2 e congela os artefatos utilizados pela F3.

### Condições obrigatórias

O empacotamento ocorre somente quando:

1. as respostas iniciais foram carregadas;
2. as revisões pendentes foram resolvidas;
3. as 150 justificativas foram aprovadas;
4. a auditoria automática foi concluída.

### Artefatos finais

O pacote contém:

```text
f2_operacional.zip
├── rationale_core.py
├── rationales_references.jsonl
├── professor_generations.jsonl.gz
├── rationale_validation.csv
└── manifest.json
```

`rationales_references.jsonl` contém as versões consolidadas de R1, R2 e R3. `professor_generations.jsonl.gz` preserva as respostas e o histórico de geração. `rationale_validation.csv` registra os critérios e resultados finais.

### Manifesto

O manifesto documenta:

- hashes de F0 e F1;
- templates utilizados;
- política de idioma;
- modelo professor;
- lotes iniciais;
- revisões aplicadas;
- quantidade por tipo;
- validações determinísticas;
- auditoria automática;
- ausência de edição manual;
- tamanhos e SHA-256 dos arquivos finais.

O manifesto registra explicitamente:

```text
revisão qualitativa manual: não realizada
auditoria final automática: realizada
```

### Verificação do ZIP

Depois da criação, o pacote é reaberto e comparado com a lista dos cinco arquivos esperados. Arquivos adicionais ou ausentes provocam erro.

### Resultado esperado

A F2 é considerada concluída somente quando o ZIP íntegro é produzido e todas as justificativas estão aprovadas.

In [11]:
# ----------------------------------------------------------
# 11.1 Verificação das condições de encerramento
# ----------------------------------------------------------

if not (
    F2_CONTINUAR_VALIDACAO
    and F2_CONTINUAR_FINALIZACAO
    and F2_READY
    and F2_AUDITORIA_APROVADA
):
    print("Bloco 11 não executado.")

    if not F2_CONTINUAR_VALIDACAO:
        print(
            "- As respostas iniciais ainda não foram "
            "liberadas no Bloco 4."
        )
    elif not F2_CONTINUAR_FINALIZACAO:
        print("- Existem revisões pendentes no Bloco 9.")
    elif not F2_READY:
        print(
            "- A validação automática ainda possui "
            "pendências."
        )
    elif not F2_AUDITORIA_APROVADA:
        print(
            "- A auditoria final automática ainda não "
            "foi concluída no Bloco 10."
        )

    print("Status: NÃO EMPACOTADO")

else:
    # ----------------------------------------------------------
    # 11.2 Limpeza de caches e preparação dos artefatos
    # ----------------------------------------------------------

    for diretorio_cache in F2_WORK_DIR.rglob("__pycache__"):
        shutil.rmtree(
            diretorio_cache,
            ignore_errors=True,
        )

    for arquivo_pyc in F2_WORK_DIR.rglob("*.pyc"):
        arquivo_pyc.unlink(missing_ok=True)

    F2_FINAL_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copyfile(
        RATIONALE_CORE_PATH,
        FINAL_RATIONALE_CORE_PATH,
    )

    salvar_jsonl(
        rationales_references,
        RATIONALES_REFERENCES_PATH,
    )

    salvar_jsonl_gzip(
        geracoes_professor,
        PROFESSOR_GENERATIONS_PATH,
    )

    validacao_export_df = validacoes_finais_df.copy()

    for coluna in [
        "ancoras_esperadas_lista",
        "ancoras_encontradas_lista",
        "ancoras_faltantes_lista",
    ]:
        if coluna in validacao_export_df.columns:
            validacao_export_df[coluna] = (
                validacao_export_df[coluna].map(
                    lambda valores: json.dumps(
                        valores,
                        ensure_ascii=False,
                        separators=(",", ":"),
                    )
                )
            )

    validacao_export_df.sort_values(
        ["tipo_justificativa", "id"]
    ).to_csv(
        RATIONALE_VALIDATION_PATH,
        index=False,
        encoding="utf-8",
    )

    arquivos_pre_manifesto = [
        FINAL_RATIONALE_CORE_PATH,
        RATIONALES_REFERENCES_PATH,
        PROFESSOR_GENERATIONS_PATH,
        RATIONALE_VALIDATION_PATH,
    ]

    if not F2_AUDITORIA_PATH.is_file():
        raise FileNotFoundError(
            "O arquivo da auditoria final automática "
            "não foi encontrado."
        )

    # ----------------------------------------------------------
    # 11.3 Construção do manifesto final
    # ----------------------------------------------------------

    manifesto_f2 = {
        "fase": FASE,
        "descricao": (
            "Geração, validação e consolidação das "
            "justificativas R1, R2 e R3 do CAMPI."
        ),
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "dataset": DATASET_ID,
        "inputs": {
            "f0_manifest_sha256": calcular_sha256(
                F0_DIR / "manifest.json"
            ),
            "f1_manifest_sha256": calcular_sha256(
                F1_DIR / "manifest.json"
            ),
            "campi_canonical_sha256": calcular_sha256(
                F0_DIR / "campi_canonical.csv"
            ),
            "ir_references_sha256": calcular_sha256(
                F1_DIR / "ir_references.jsonl"
            ),
            "ir_schema_sha256": calcular_sha256(
                F1_DIR / "ir_schema.json"
            ),
            "templates_f0_sha256": {
                chave: calcular_sha256(caminho)
                for chave, caminho in TEMPLATE_PATHS.items()
            },
        },
        "prompting": {
            "instruction_language": "Português",
            "rationale_language": "English",
            "operational_template_sha256": {
                chave: calcular_sha256_texto(template)
                for chave, template in templates_f2.items()
            },
            "structured_revision_feedback": True,
            "revision_placeholders": sorted(
                placeholders_esperados["REVISAO"]
            ),
            "interactive_response_gate": True,
            "interactive_revision_gate": True,
            "interactive_final_audit_gate": False,
        },
        "teacher": {
            "model": TEACHER_MODEL,
            "generation_mode": "external_manual_import",
            "batch_size": BATCH_SIZE,
            "initial_batches": EXPECTED_GENERATION_BATCHES,
            "initial_responses_loaded": (
                EXPECTED_GENERATION_BATCHES
            ),
            "revisions_applied": len(
                revisoes_aplicadas
            ),
        },
        "rationales": {
            "version": RATIONALE_VERSION,
            "language": "English",
            "types": list(RATIONALE_TYPES),
            "examples": EXPECTED_EXAMPLES,
            "total": EXPECTED_RATIONALES,
            "r1_approved": int(
                (
                    validacoes_finais_df[
                        "tipo_justificativa"
                    ].eq("R1")
                    & validacoes_finais_df[
                        "aprovada_finalmente"
                    ]
                ).sum()
            ),
            "r2_approved": int(
                (
                    validacoes_finais_df[
                        "tipo_justificativa"
                    ].eq("R2")
                    & validacoes_finais_df[
                        "aprovada_finalmente"
                    ]
                ).sum()
            ),
            "r3_approved": int(
                (
                    validacoes_finais_df[
                        "tipo_justificativa"
                    ].eq("R3")
                    & validacoes_finais_df[
                        "aprovada_finalmente"
                    ]
                ).sum()
            ),
            "all_approved": bool(
                validacoes_finais_df[
                    "aprovada_finalmente"
                ].all()
            ),
        },
        "validation": {
            "module": "rationale_core.py",
            "deterministic": True,
            "checks": [
                "estrutura da resposta",
                "IDs e quantidade por lote",
                "extensão da justificativa",
                "idioma inglês",
                (
                    "rejeição de justificativas "
                    "em português"
                ),
                (
                    "ausência de metalinguagem "
                    "e Markdown"
                ),
                (
                    "operadores e flexões verbais "
                    "equivalentes"
                ),
                "valores de escopo",
                "direção dos escopos de rota",
                "campos e itens das operações",
                "restrições temporais",
                "valores formais em chamadas Nile",
                (
                    "identificação explícita das "
                    "âncoras ausentes"
                ),
                (
                    "feedback estruturado para "
                    "revisão seletiva"
                ),
            ],
            "semantic_proof": False,
            "qualitative_review_required": False,
            "manual_qualitative_review_performed": False,
            "automatic_final_audit_required": True,
            "automatic_final_audit_completed": bool(
                F2_AUDITORIA_APROVADA
            ),
            "automatic_final_audit_completed_at_utc": (
                F2_AUDITORIA_CONCLUIDA_EM_UTC
            ),
            "automatic_final_audit_file": (
                F2_AUDITORIA_PATH.name
            ),
            "automatic_final_audit_sha256": calcular_sha256(
                F2_AUDITORIA_PATH
            ),
            "audited_examples": int(
                len(auditoria_df)
            ),
            "audited_rationales": int(
                total_justificativas_auditadas
            ),
            "approved_rationales": int(
                total_justificativas_aprovadas
            ),
        },
        "method": {
            "teacher_model_used": True,
            "student_model_used": False,
            "ir_modified": False,
            "nile_modified": False,
            "manual_rationale_editing": False,
            "manual_qualitative_review": False,
            "automatic_final_audit": True,
            "f0_artifacts_modified": False,
            "selective_teacher_revision": True,
        },
        "environment": {
            "python": platform.python_version(),
            "platform": platform.platform(),
            "pandas": pd.__version__,
        },
        "files": [
            {
                "arquivo": caminho.name,
                "tamanho_bytes": caminho.stat().st_size,
                "sha256": calcular_sha256(caminho),
            }
            for caminho in arquivos_pre_manifesto
        ],
    }

    salvar_json(
        manifesto_f2,
        MANIFEST_PATH,
    )

    # ----------------------------------------------------------
    # 11.4 Criação e verificação do pacote final
    # ----------------------------------------------------------

    arquivos_finais = [
        FINAL_RATIONALE_CORE_PATH,
        RATIONALES_REFERENCES_PATH,
        PROFESSOR_GENERATIONS_PATH,
        RATIONALE_VALIDATION_PATH,
        MANIFEST_PATH,
    ]

    if len(arquivos_finais) != EXPECTED_FINAL_FILES:
        raise ValueError(
            "Quantidade inesperada de arquivos finais "
            "da F2."
        )

    if not all(
        caminho.is_file()
        for caminho in arquivos_finais
    ):
        raise FileNotFoundError(
            "Um ou mais artefatos finais da F2 "
            "não foram criados."
        )

    criar_zip(
        F2_ZIP_PATH,
        [
            (caminho, caminho.name)
            for caminho in arquivos_finais
        ],
    )

    with zipfile.ZipFile(
        F2_ZIP_PATH,
        "r",
    ) as pacote:
        nomes_zip = sorted(
            pacote.namelist()
        )

    if len(nomes_zip) != EXPECTED_FINAL_FILES:
        raise ValueError(
            "O ZIP final não contém exatamente "
            "cinco arquivos."
        )

    nomes_esperados_zip = sorted(
        caminho.name
        for caminho in arquivos_finais
    )

    if nomes_zip != nomes_esperados_zip:
        raise ValueError(
            "O conteúdo do ZIP final difere "
            "dos arquivos esperados."
        )

    resumo_final_df = pd.DataFrame([{
        "fase": FASE,
        "dataset": DATASET_ID,
        "total_exemplos": EXPECTED_EXAMPLES,
        "total_justificativas": EXPECTED_RATIONALES,
        "aprovadas": int(
            validacoes_finais_df[
                "aprovada_finalmente"
            ].sum()
        ),
        "geracoes_professor": len(
            geracoes_professor
        ),
        "revisoes_aplicadas": len(
            revisoes_aplicadas
        ),
        "auditoria_final_automatica": (
            F2_AUDITORIA_APROVADA
        ),
        "arquivos_no_zip": len(
            nomes_zip
        ),
        "zip": str(
            F2_ZIP_PATH
        ),
        "status": "approved",
    }])

    exibir_tabela(
        resumo_final_df,
        titulo="Resumo final da F2",
    )

    print("Bloco 11 concluído")
    print(f"ZIP gerado: {F2_ZIP_PATH}")
    print(
        "Tamanho do ZIP: "
        f"{F2_ZIP_PATH.stat().st_size:,} bytes"
    )
    print("F2 concluída com sucesso")
    print("Status: OK")

fase,conjunto de dados,exemplos,justificativas,aprovadas,gerações do professor,revisões aplicadas,auditoria final automatica,arquivos no ZIP,ZIP,status
F2,CAMPI,50,150,150,7,1,sim,5,/kaggle/working/f2_operacional.zip,aprovado


Bloco 11 concluído
ZIP gerado: /kaggle/working/f2_operacional.zip
Tamanho do ZIP: 39,696 bytes
F2 concluída com sucesso
Status: OK
